# Classificazione di patologie renali da immagini TC

Pipeline sperimentale della tesi di Aurora Macali: **YOLO11n per la localizzazione dei reni**, generazione di due pannelli 224×224 e confronto tra **ResNet50 baseline** e **ResNet50 a due rami con encoder condiviso**.

Il notebook è progettato per Google Colab con GPU NVIDIA Tesla T4. Eseguire le celle dall'alto verso il basso senza cambiare ordine.


## 1. Ambiente e configurazione


In [ ]:
!pip install -q ultralytics==8.4.117 seaborn pyyaml opencv-python-headless


In [ ]:
import ultralytics
import torch

ultralytics.checks()

print("PyTorch:", torch.__version__)
print("GPU disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import torch

SEED = 42
COLAB_ROOT = Path("/content")
DRIVE_ROOT = Path("/content/drive/MyDrive")

ANNOTATED_DATASET_ZIP = DRIVE_ROOT / "Kidney_YOLO_Group_Split.zip"
YOLO_WEIGHTS_SOURCE = DRIVE_ROOT / "kidney_yolo11n_best.zip"
FULL_DATASET_ZIP = DRIVE_ROOT / "full_dataset_kidney_kaggle.zip"

TRAIN_YOLO_FROM_SCRATCH = False

RESULTS_ROOT = DRIVE_ROOT / "Kidney_Thesis_Results"
YOLO_ANNOTATED_RESULTS = RESULTS_ROOT / "YOLO_Annotated"
YOLO_TEST_RESULTS = RESULTS_ROOT / "YOLO_Test_conf025"
BASELINE_RESULTS = RESULTS_ROOT / "ResNet50_Baseline"
TWO_BRANCH_RESULTS_ROOT = RESULTS_ROOT / "ResNet50_TwoBranch"
FULL_DATASET_RESULTS = RESULTS_ROOT / "Full_Dataset"

for directory in (
    RESULTS_ROOT,
    YOLO_ANNOTATED_RESULTS,
    YOLO_TEST_RESULTS,
    BASELINE_RESULTS,
    TWO_BRANCH_RESULTS_ROOT,
    FULL_DATASET_RESULTS,
):
    directory.mkdir(parents=True, exist_ok=True)

YOLO_DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Cartella risultati:", RESULTS_ROOT)
print("Device YOLO:", YOLO_DEVICE)


### File di input attesi in `MyDrive`

- `Kidney_YOLO_Group_Split.zip`: sottoinsieme annotato, etichette YOLO e manifest group-aware;
- `kidney_yolo11n_best.zip`: checkpoint selezionato sul validation set;
- `full_dataset_kidney_kaggle.zip`: dataset Kaggle completo organizzato per gruppi.


In [ ]:
from pathlib import Path
import shutil
import zipfile

ZIP_DRIVE = ANNOTATED_DATASET_ZIP
WEIGHTS_DOWNLOADED = YOLO_WEIGHTS_SOURCE
ZIP_LOCAL = COLAB_ROOT / "Kidney_YOLO_Group_Split.zip"
WEIGHTS_LOCAL = COLAB_ROOT / "kidney_yolo11n_best.pt"
DATASET_DIR = COLAB_ROOT / "Kidney_YOLO_Group_Split"

assert ZIP_DRIVE.exists(), f"Dataset annotato non trovato: {ZIP_DRIVE}"
shutil.copy2(ZIP_DRIVE, ZIP_LOCAL)

if not TRAIN_YOLO_FROM_SCRATCH:
    assert WEIGHTS_DOWNLOADED.exists(), f"Checkpoint YOLO non trovato: {WEIGHTS_DOWNLOADED}"
    shutil.copy2(WEIGHTS_DOWNLOADED, WEIGHTS_LOCAL)

print("Dataset annotato:", ZIP_DRIVE)
print("Archivio locale:", ZIP_LOCAL)
if not TRAIN_YOLO_FROM_SCRATCH:
    print("Checkpoint locale:", WEIGHTS_LOCAL)


In [ ]:
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)


In [ ]:
with zipfile.ZipFile(ZIP_LOCAL, "r") as archive:
    archive.extractall(COLAB_ROOT)

MANIFEST = DATASET_DIR / "metadata" / "split_manifest.csv"

print("Dataset trovato:", DATASET_DIR.exists())
print("Manifest trovato:", MANIFEST.exists())
print("Pesi locali trovati:", WEIGHTS_LOCAL.exists() if not TRAIN_YOLO_FROM_SCRATCH else "da addestrare")


## 2. Addestramento di YOLO11n


In [ ]:
from ultralytics import YOLO

if TRAIN_YOLO_FROM_SCRATCH:
    training_model = YOLO("yolo11n.pt")
    training_result = training_model.train(
        data=str(DATASET_DIR / "data.yaml"),
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        seed=SEED,
        deterministic=True,
        device=YOLO_DEVICE,
        project=str(RESULTS_ROOT / "YOLO_Training"),
        name="yolo11n_kidney",
        exist_ok=True,
    )
    WEIGHTS_LOCAL = Path(training_result.save_dir) / "weights" / "best.pt"

assert WEIGHTS_LOCAL.exists(), f"Checkpoint YOLO non disponibile: {WEIGHTS_LOCAL}"
print("Checkpoint usato:", WEIGHTS_LOCAL)


## 3. Valutazione di YOLO11n sul test annotato


In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO


DATA_YAML = DATASET_DIR / "data.yaml"

data_config = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "kidney"
    }
}

with open(
    DATA_YAML,
    "w",
    encoding="utf-8"
) as file:
    yaml.safe_dump(
        data_config,
        file,
        sort_keys=False
    )

print(DATA_YAML.read_text())


yolo_model = YOLO(
    str(WEIGHTS_LOCAL)
)

YOLO_EVALUATION_DIR = YOLO_TEST_RESULTS

test_metrics_yolo = yolo_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=YOLO_DEVICE,
    conf=0.25,
    iou=0.60,
    plots=True,
    project=str(YOLO_EVALUATION_DIR.parent),
    name="YOLO_Test_conf025",
    exist_ok=True
)

print("\nRISULTATI YOLO SUL TEST SET")
print("=" * 40)
print(
    "Precision:",
    f"{test_metrics_yolo.box.mp:.4f}"
)
print(
    "Recall:",
    f"{test_metrics_yolo.box.mr:.4f}"
)
print(
    "mAP@0.50:",
    f"{test_metrics_yolo.box.map50:.4f}"
)
print(
    "mAP@0.50:0.95:",
    f"{test_metrics_yolo.box.map:.4f}"
)


In [ ]:
import json
import matplotlib.pyplot as plt


metric_names = [
    "Precision",
    "Recall",
    "mAP@0.50",
    "mAP@0.50:0.95"
]

metric_values = [
    float(test_metrics_yolo.box.mp),
    float(test_metrics_yolo.box.mr),
    float(test_metrics_yolo.box.map50),
    float(test_metrics_yolo.box.map)
]

metrics_dictionary = dict(
    zip(
        metric_names,
        metric_values
    )
)

with open(
    YOLO_EVALUATION_DIR / "test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metrics_dictionary,
        file,
        indent=2
    )


plt.style.use("seaborn-v0_8-whitegrid")

fig, axis = plt.subplots(
    figsize=(9, 6)
)

colors = [
    "#4C78A8",
    "#F58518",
    "#54A24B",
    "#E45756"
]

bars = axis.bar(
    metric_names,
    metric_values,
    color=colors,
    width=0.65
)

axis.set_title(
    "Prestazioni di YOLO11n sul test set",
    fontsize=15,
    fontweight="bold"
)

axis.set_ylabel("Valore")
axis.set_ylim(0, 1.08)

axis.bar_label(
    bars,
    labels=[
        f"{value:.3f}"
        for value in metric_values
    ],
    padding=4,
    fontsize=11
)

plt.tight_layout()

plt.savefig(
    YOLO_EVALUATION_DIR
    / "yolo_test_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    YOLO_EVALUATION_DIR
    / "yolo_test_metrics.pdf",
    bbox_inches="tight"
)

plt.show()


## 4. Generazione dei crop sul sottoinsieme annotato


In [ ]:
from pathlib import Path
import shutil
import json

import pandas as pd
import numpy as np

from PIL import Image
from ultralytics import YOLO



CONFIDENCE = 0.25
IOU = 0.70

MARGIN_RATIO = 0.05

KIDNEY_SIZE = 224

FINAL_WIDTH = KIDNEY_SIZE * 2
FINAL_HEIGHT = KIDNEY_SIZE



def create_kidney_panel(image, box):
    """
    Ritaglia un singolo rene dalla box YOLO, aggiunge un piccolo
    margine, mantiene le proporzioni e lo centra in un pannello
    nero 224×224.

    Restituisce:
        panel: immagine 224×224
        coordinates: coordinate effettive del crop
    """

    image_width, image_height = image.size

    x1, y1, x2, y2 = map(float, box)

    box_width = x2 - x1
    box_height = y2 - y1

    if box_width <= 0 or box_height <= 0:
        return None, None

    margin_x = box_width * MARGIN_RATIO
    margin_y = box_height * MARGIN_RATIO

    x1 = max(
        0,
        int(np.floor(x1 - margin_x))
    )

    y1 = max(
        0,
        int(np.floor(y1 - margin_y))
    )

    x2 = min(
        image_width,
        int(np.ceil(x2 + margin_x))
    )

    y2 = min(
        image_height,
        int(np.ceil(y2 + margin_y))
    )

    if x2 <= x1 or y2 <= y1:
        return None, None

    kidney_crop = image.crop(
        (x1, y1, x2, y2)
    )

    kidney_crop.thumbnail(
        (KIDNEY_SIZE, KIDNEY_SIZE),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        mode="RGB",
        size=(KIDNEY_SIZE, KIDNEY_SIZE),
        color=(0, 0, 0)
    )

    paste_x = (
        KIDNEY_SIZE - kidney_crop.width
    ) // 2

    paste_y = (
        KIDNEY_SIZE - kidney_crop.height
    ) // 2

    panel.paste(
        kidney_crop,
        (paste_x, paste_y)
    )

    coordinates = {
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,
        "width": x2 - x1,
        "height": y2 - y1
    }

    return panel, coordinates



model = YOLO(str(WEIGHTS_LOCAL))

OUTPUT_DIR = COLAB_ROOT / "Kidney_Classification"

CROPS_DIR = OUTPUT_DIR / "crops"

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CROPS_DIR.mkdir(
    parents=True,
    exist_ok=True
)



manifest = pd.read_csv(MANIFEST)

required_columns = {
    "category",
    "group",
    "filename",
    "split",
    "boxes"
}

assert required_columns.issubset(manifest.columns), (
    "Nel manifest mancano alcune colonne.\n"
    f"Colonne trovate: {manifest.columns.tolist()}"
)

valid_categories = {
    "Normal",
    "Cyst",
    "Stone",
    "Tumor"
}

valid_splits = {
    "train",
    "val",
    "test"
}

assert set(manifest["category"]).issubset(
    valid_categories
), (
    "Categorie inattese: "
    f"{set(manifest['category'])}"
)

assert set(manifest["split"]).issubset(
    valid_splits
), (
    "Split inattesi: "
    f"{set(manifest['split'])}"
)



records = []
anomalies = []

for index, row in manifest.iterrows():

    split = str(row["split"])
    category = str(row["category"])
    filename = str(row["filename"])
    group = row["group"]
    expected_boxes = int(row["boxes"])

    image_path = (
        DATASET_DIR
        / "images"
        / split
        / filename
    )


    if not image_path.exists():
        anomalies.append({
            "filename": filename,
            "split": split,
            "category": category,
            "reason": "image_missing",
            "expected": expected_boxes,
            "detected": 0
        })

        continue


    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=CONFIDENCE,
        iou=IOU,
        device=YOLO_DEVICE,
        verbose=False
    )[0]

    image = Image.open(
        image_path
    ).convert("RGB")

    original_width, original_height = image.size


    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):
        anomalies.append({
            "filename": filename,
            "split": split,
            "category": category,
            "reason": "no_detection",
            "expected": expected_boxes,
            "detected": 0
        })

        continue

    all_boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    all_confidences = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    detected_count = len(all_boxes)


    confidence_order = np.argsort(
        all_confidences
    )[::-1]

    selected_indices = confidence_order[:2]

    selected_boxes = all_boxes[
        selected_indices
    ]

    selected_confidences = all_confidences[
        selected_indices
    ]


    horizontal_order = np.argsort(
        (
            selected_boxes[:, 0]
            + selected_boxes[:, 2]
        ) / 2
    )

    selected_boxes = selected_boxes[
        horizontal_order
    ]

    selected_confidences = selected_confidences[
        horizontal_order
    ]


    kidney_panels = []
    valid_boxes = []
    valid_confidences = []

    for box, confidence in zip(
        selected_boxes,
        selected_confidences
    ):
        panel, coordinates = create_kidney_panel(
            image=image,
            box=box
        )

        if panel is None:
            anomalies.append({
                "filename": filename,
                "split": split,
                "category": category,
                "reason": "invalid_kidney_box",
                "expected": expected_boxes,
                "detected": detected_count
            })

            continue

        kidney_panels.append(panel)
        valid_boxes.append(coordinates)
        valid_confidences.append(
            float(confidence)
        )

    if len(kidney_panels) == 0:
        anomalies.append({
            "filename": filename,
            "split": split,
            "category": category,
            "reason": "no_valid_crop",
            "expected": expected_boxes,
            "detected": detected_count
        })

        continue

    real_kidneys_used = len(kidney_panels)


    if len(kidney_panels) == 1:
        black_panel = Image.new(
            mode="RGB",
            size=(KIDNEY_SIZE, KIDNEY_SIZE),
            color=(0, 0, 0)
        )

        kidney_panels.append(black_panel)

    kidney_panels = kidney_panels[:2]


    combined_image = Image.new(
        mode="RGB",
        size=(FINAL_WIDTH, FINAL_HEIGHT),
        color=(0, 0, 0)
    )

    combined_image.paste(
        kidney_panels[0],
        (0, 0)
    )

    combined_image.paste(
        kidney_panels[1],
        (KIDNEY_SIZE, 0)
    )


    destination_dir = (
        CROPS_DIR
        / split
        / category
    )

    destination_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    crop_name = (
        f"{Path(filename).stem}"
        "_kidneys_separate.jpg"
    )

    crop_path = (
        destination_dir
        / crop_name
    )

    combined_image.save(
        crop_path,
        quality=95
    )


    records.append({
        "original_filename": filename,
        "crop_filename": crop_name,
        "crop_path": str(
            crop_path.relative_to(OUTPUT_DIR)
        ),
        "category": category,
        "group": group,
        "split": split,
        "expected_gt_boxes": expected_boxes,
        "detections_found": detected_count,
        "detections_selected": len(selected_indices),
        "detections_used": real_kidneys_used,
        "black_panels": 2 - real_kidneys_used,
        "mean_confidence": float(
            np.mean(valid_confidences)
        ),
        "min_confidence": float(
            np.min(valid_confidences)
        ),
        "selected_boxes": json.dumps(
            valid_boxes,
            ensure_ascii=False
        ),
        "original_width": original_width,
        "original_height": original_height,
        "output_width": FINAL_WIDTH,
        "output_height": FINAL_HEIGHT
    })


    if detected_count != expected_boxes:
        anomalies.append({
            "filename": filename,
            "split": split,
            "category": category,
            "reason": "detection_count_mismatch",
            "expected": expected_boxes,
            "detected": detected_count
        })

    if (index + 1) % 50 == 0:
        print(
            f"Elaborate {index + 1}/"
            f"{len(manifest)} immagini"
        )



classification_manifest = pd.DataFrame(
    records
)

anomaly_columns = [
    "filename",
    "split",
    "category",
    "reason",
    "expected",
    "detected"
]

anomalies_df = pd.DataFrame(
    anomalies,
    columns=anomaly_columns
)

classification_manifest.to_csv(
    OUTPUT_DIR / "classification_manifest.csv",
    index=False
)

anomalies_df.to_csv(
    OUTPUT_DIR / "anomalies.csv",
    index=False
)



config = {
    "detector": "YOLO11n best.pt",
    "confidence_threshold": CONFIDENCE,
    "nms_iou_threshold": IOU,
    "margin_ratio_per_kidney": MARGIN_RATIO,
    "kidney_panel_size": [
        KIDNEY_SIZE,
        KIDNEY_SIZE
    ],
    "final_image_size": [
        FINAL_WIDTH,
        FINAL_HEIGHT
    ],
    "strategy": (
        "Two independent kidney crops, ordered from image-left "
        "to image-right and placed side by side"
    ),
    "single_detection_strategy": (
        "Detected kidney crop plus one black panel"
    ),
    "selection_strategy": (
        "At most the two highest-confidence YOLO detections"
    ),
    "note": (
        "Ogni rene viene ritagliato separatamente. "
        "Non viene conservata la regione anatomica compresa "
        "tra i due reni."
    )
}

with open(
    OUTPUT_DIR / "crop_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        config,
        file,
        indent=2,
        ensure_ascii=False
    )



print("\nGenerazione completata")
print("Immagini originali:", len(manifest))
print("Immagini create:", len(classification_manifest))
print("Segnalazioni:", len(anomalies_df))

if len(classification_manifest) > 0:
    print(
        "Immagini con due reni:",
        int(
            (
                classification_manifest[
                    "detections_used"
                ] == 2
            ).sum()
        )
    )

    print(
        "Immagini con un rene + nero:",
        int(
            (
                classification_manifest[
                    "black_panels"
                ] == 1
            ).sum()
        )
    )

print(
    "Dimensione immagini finali:",
    f"{FINAL_WIDTH}×{FINAL_HEIGHT}"
)


In [ ]:
summary = (
    classification_manifest
    .groupby(["split", "category"])
    .size()
    .unstack(fill_value=0)
)

print("Numero di crop per split e categoria:")
display(summary)

print("\nTotale immagini originali:", len(manifest))
print("Totale crop creati:", len(classification_manifest))

missing_detections = anomalies_df[
    anomalies_df["reason"] == "no_detection"
]

count_mismatches = anomalies_df[
    anomalies_df["reason"] == "detection_count_mismatch"
]

print("Immagini senza rilevamenti:", len(missing_detections))
print("Immagini con numero di box diverso:", len(count_mismatches))

print("\nPrime righe del manifest:")
display(classification_manifest.head())

print("\nPrime anomalie:")
display(anomalies_df.head(20))


### 4.1 Analisi delle anomalie e figure qualitative


In [ ]:
import pandas as pd

unique_anomalies = (
    anomalies_df
    .drop_duplicates(
        subset=[
            "filename",
            "split",
            "category",
            "reason"
        ]
    )
)

anomaly_distribution = pd.crosstab(
    unique_anomalies["category"],
    unique_anomalies["split"],
    margins=True,
    margins_name="Totale"
)

desired_columns = [
    column
    for column in [
        "train",
        "val",
        "test",
        "Totale"
    ]
    if column in anomaly_distribution.columns
]

anomaly_distribution = (
    anomaly_distribution[
        desired_columns
    ]
)

display(anomaly_distribution)

anomaly_distribution.to_csv(
    YOLO_ANNOTATED_RESULTS
    / "anomaly_distribution_by_class_split.csv"
)


In [ ]:
mismatch_cases = (
    anomalies_df.loc[
        anomalies_df["reason"]
        == "detection_count_mismatch"
    ]
    .drop_duplicates(
        subset=["filename", "split"]
    )
)

missing_detection_cases = mismatch_cases.loc[
    mismatch_cases["detected"]
    < mismatch_cases["expected"]
]

additional_detection_cases = mismatch_cases.loc[
    mismatch_cases["detected"]
    > mismatch_cases["expected"]
]

print(
    "Casi con detection mancanti:",
    len(missing_detection_cases)
)

print(
    "Casi con detection aggiuntive:",
    len(additional_detection_cases)
)

display(missing_detection_cases.head())
display(additional_detection_cases.head())


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

from PIL import Image
from ultralytics import YOLO

detector = YOLO(
    str(WEIGHTS_LOCAL)
)


def load_ground_truth_boxes(
    label_path,
    image_width,
    image_height
):
    boxes = []

    if not label_path.exists():
        return boxes

    lines = (
        label_path
        .read_text()
        .strip()
        .splitlines()
    )

    for line in lines:
        if not line.strip():
            continue

        values = list(
            map(
                float,
                line.split()
            )
        )

        class_id, x_center, y_center, width, height = (
            values[:5]
        )

        x1 = (
            x_center - width / 2
        ) * image_width

        y1 = (
            y_center - height / 2
        ) * image_height

        x2 = (
            x_center + width / 2
        ) * image_width

        y2 = (
            y_center + height / 2
        ) * image_height

        boxes.append(
            [x1, y1, x2, y2]
        )

    return boxes


def create_anomaly_figure(
    anomaly_row,
    title,
    output_path
):
    split = anomaly_row["split"]
    filename = anomaly_row["filename"]

    image_path = (
        DATASET_DIR
        / "images"
        / split
        / filename
    )

    label_path = (
        DATASET_DIR
        / "labels"
        / split
        / f"{Path(filename).stem}.txt"
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    image_width, image_height = image.size

    ground_truth_boxes = (
        load_ground_truth_boxes(
            label_path,
            image_width,
            image_height
        )
    )

    result = detector.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        iou=0.70,
        device=YOLO_DEVICE,
        verbose=False
    )[0]

    if (
        result.boxes is not None
        and len(result.boxes) > 0
    ):
        predicted_boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        confidences = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )
    else:
        predicted_boxes = np.empty(
            (0, 4)
        )

        confidences = np.empty(0)

    figure, axis = plt.subplots(
        figsize=(8, 8)
    )

    axis.imshow(image)

    for index, box in enumerate(
        ground_truth_boxes
    ):
        x1, y1, x2, y2 = box

        rectangle = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2.5,
            edgecolor="lime",
            facecolor="none"
        )

        axis.add_patch(rectangle)

        axis.text(
            x1,
            max(0, y1 - 6),
            "Ground truth",
            color="lime",
            fontsize=9,
            backgroundcolor="black"
        )

    for box, confidence in zip(
        predicted_boxes,
        confidences
    ):
        x1, y1, x2, y2 = box

        rectangle = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
            linestyle="--"
        )

        axis.add_patch(rectangle)

        axis.text(
            x1,
            y2 + 6,
            f"Predizione: {confidence:.2f}",
            color="red",
            fontsize=9,
            backgroundcolor="black"
        )

    axis.set_title(title)
    axis.axis("off")

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()


In [ ]:
ANOMALY_FIGURES_DIR = (
    YOLO_ANNOTATED_RESULTS
    / "detection_anomalies"
)

ANOMALY_FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not missing_detection_cases.empty:
    missing_example = (
        missing_detection_cases.iloc[0]
    )

    create_anomaly_figure(
        missing_example,
        (
            "Esempio di detection mancante "
            "rispetto alla ground truth"
        ),
        (
            ANOMALY_FIGURES_DIR
            / "missing_detection.png"
        )
    )

if not additional_detection_cases.empty:
    additional_example = (
        additional_detection_cases.iloc[0]
    )

    create_anomaly_figure(
        additional_example,
        (
            "Esempio di detection aggiuntiva "
            "rispetto alla ground truth"
        ),
        (
            ANOMALY_FIGURES_DIR
            / "additional_detection.png"
        )
    )


In [ ]:
created_filenames = set(
    classification_manifest["original_filename"]
)

missing_crops = manifest[
    ~manifest["filename"].isin(created_filenames)
]

display(
    missing_crops[
        ["filename", "category", "group", "split", "boxes"]
    ]
)


In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pathlib import Path
from PIL import Image


example_candidates = (
    classification_manifest.loc[
        (
            classification_manifest["split"]
            == "test"
        )
        & (
            classification_manifest[
                "detections_used"
            ]
            == 2
        )
    ]
    .sort_values(
        "mean_confidence",
        ascending=False
    )
)

assert not example_candidates.empty, (
    "Nessun esempio con due reni trovato."
)

example = example_candidates.iloc[0]

original_image_path = (
    DATASET_DIR
    / "images"
    / example["split"]
    / example["original_filename"]
)

combined_crop_path = (
    OUTPUT_DIR
    / example["crop_path"]
)

assert original_image_path.exists(), (
    f"Immagine non trovata: {original_image_path}"
)

assert combined_crop_path.exists(), (
    f"Crop non trovato: {combined_crop_path}"
)


original_image = Image.open(
    original_image_path
).convert("RGB")

combined_crop = Image.open(
    combined_crop_path
).convert("RGB")

selected_boxes = json.loads(
    example["selected_boxes"]
)


figure, axes = plt.subplots(
    1,
    3,
    figsize=(17, 6)
)


axes[0].imshow(original_image)
axes[0].set_title(
    "(a) Immagine TC originale",
    fontsize=13
)
axes[0].axis("off")


axes[1].imshow(original_image)

for index, box in enumerate(
    selected_boxes,
    start=1
):
    x1 = box["x1"]
    y1 = box["y1"]
    x2 = box["x2"]
    y2 = box["y2"]

    rectangle = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )

    axes[1].add_patch(rectangle)

    axes[1].text(
        x1,
        max(0, y1 - 5),
        f"Rene {index}",
        color="yellow",
        fontsize=10,
        backgroundcolor="black"
    )

axes[1].set_title(
    "(b) Regioni individuate da YOLO11n",
    fontsize=13
)
axes[1].axis("off")


axes[2].imshow(combined_crop)

axes[2].axvline(
    x=224,
    color="red",
    linestyle="--",
    linewidth=2
)

axes[2].text(
    112,
    15,
    "Pannello 1",
    color="yellow",
    fontsize=11,
    ha="center",
    backgroundcolor="black"
)

axes[2].text(
    336,
    15,
    "Pannello 2",
    color="yellow",
    fontsize=11,
    ha="center",
    backgroundcolor="black"
)

axes[2].set_title(
    "(c) Input finale 448 × 224 pixel",
    fontsize=13
)
axes[2].axis("off")

plt.tight_layout(
    rect=[0, 0, 1, 0.98]
)

plt.tight_layout()

PIPELINE_FIGURE_PATH = (
    YOLO_ANNOTATED_RESULTS
    / "yolo_crop_pipeline.png"
)

plt.savefig(
    PIPELINE_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Esempio utilizzato:", example["original_filename"])
print("Figura salvata in:", PIPELINE_FIGURE_PATH)


In [ ]:
import matplotlib.pyplot as plt

sample_size = min(12, len(classification_manifest))

sample = classification_manifest.sample(
    n=sample_size,
    random_state=42
)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for axis in axes:
    axis.axis("off")

for axis, (_, row) in zip(axes, sample.iterrows()):
    crop_path = OUTPUT_DIR / row["crop_path"]
    crop_image = Image.open(crop_path)

    axis.imshow(crop_image, cmap="gray")
    axis.set_title(
        f'{row["category"]} | {row["split"]}\n'
        f'confidence: {row["mean_confidence"]:.2f}'
    )
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import math
import matplotlib.pyplot as plt

from PIL import Image


to_check = classification_manifest[
    (classification_manifest["detections_used"] == 1)
    |
    (classification_manifest["min_confidence"] < 0.50)
].copy()

to_check = to_check.sort_values(
    by=["detections_used", "min_confidence"],
    ascending=[True, True]
)

print("Crop da controllare:", len(to_check))

if len(to_check) == 0:
    print(
        "Nessun crop con un solo rene "
        "o confidence inferiore a 0.50."
    )

else:
    columns = 4
    rows = math.ceil(
        len(to_check) / columns
    )

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, rows * 3.2),
        squeeze=False
    )

    axes = axes.flatten()

    for axis in axes:
        axis.axis("off")

    for axis, (_, row) in zip(
        axes,
        to_check.iterrows()
    ):
        image_path = (
            OUTPUT_DIR
            / row["crop_path"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        axis.imshow(image)

        axis.set_title(
            f'{row["original_filename"]}\n'
            f'{row["category"]} | {row["split"]}\n'
            f'trovati: {row["detections_found"]} | '
            f'usati: {row["detections_used"]}\n'
            f'nero: {row["black_panels"]} | '
            f'min conf: {row["min_confidence"]:.2f}',
            fontsize=9
        )

        axis.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
DRIVE_OUTPUT = RESULTS_ROOT / "Classification_Annotated"

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

archive_path = shutil.make_archive(
    "/content/Kidney_Classification",
    "zip",
    root_dir="/content",
    base_dir="Kidney_Classification"
)

final_zip = DRIVE_OUTPUT / "Kidney_Classification.zip"

shutil.copy2(archive_path, final_zip)

assert final_zip.exists(), "Lo ZIP finale non è stato salvato"

print("ZIP salvato correttamente:")
print(final_zip)
print(
    "Dimensione:",
    round(final_zip.stat().st_size / (1024 ** 2), 2),
    "MB"
)


## 5. Estensione della pipeline al dataset completo


In [ ]:
from pathlib import Path

KAGGLE_ZIP_DRIVE = FULL_DATASET_ZIP

assert KAGGLE_ZIP_DRIVE.exists(), (
    f"Archivio non trovato: {KAGGLE_ZIP_DRIVE}"
)

print("Archivio trovato:", KAGGLE_ZIP_DRIVE)
print(
    "Dimensione:",
    round(
        KAGGLE_ZIP_DRIVE.stat().st_size
        / (1024 ** 3),
        2
    ),
    "GB"
)


In [ ]:
from pathlib import Path
import shutil
import zipfile

KAGGLE_ZIP_LOCAL = COLAB_ROOT / "full_dataset_kidney_kaggle.zip"

KAGGLE_EXTRACT_DIR = COLAB_ROOT / "full_dataset_kidney_kaggle"

if KAGGLE_EXTRACT_DIR.exists():
    shutil.rmtree(KAGGLE_EXTRACT_DIR)

print("Copia dello ZIP in corso...")

shutil.copy2(
    KAGGLE_ZIP_DRIVE,
    KAGGLE_ZIP_LOCAL
)

print("Estrazione in corso...")

KAGGLE_EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with zipfile.ZipFile(
    KAGGLE_ZIP_LOCAL,
    "r"
) as archive:
    archive.extractall(
        KAGGLE_EXTRACT_DIR
    )

print("Estrazione completata:", KAGGLE_EXTRACT_DIR)


In [ ]:
from pathlib import Path

grouped_candidates = [
    path
    for path in KAGGLE_EXTRACT_DIR.rglob("*")
    if (
        path.is_dir()
        and path.name.lower() == "grouped images"
    )
]

assert grouped_candidates, (
    "Cartella 'Grouped images' non trovata "
    "nell'archivio estratto"
)

KAGGLE_GROUPED_DIR = grouped_candidates[0]

print("Dataset principale:", KAGGLE_GROUPED_DIR)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
}

categories = [
    path
    for path in KAGGLE_GROUPED_DIR.iterdir()
    if path.is_dir()
]

total_images = 0

for category_directory in sorted(categories):

    groups = [
        path
        for path in category_directory.iterdir()
        if path.is_dir()
    ]

    images = [
        path
        for path in category_directory.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]

    total_images += len(images)

    print(
        f"{category_directory.name:8s} | "
        f"gruppi: {len(groups):3d} | "
        f"immagini: {len(images):5d}"
    )

print("Totale immagini:", total_images)


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

YOLO_DATASET_ZIP = ANNOTATED_DATASET_ZIP

assert YOLO_DATASET_ZIP.exists(), (
    f"ZIP YOLO non trovato: {YOLO_DATASET_ZIP}"
)

with zipfile.ZipFile(
    YOLO_DATASET_ZIP,
    "r"
) as archive:

    manifest_members = [
        name
        for name in archive.namelist()
        if name.endswith(
            "metadata/split_manifest.csv"
        )
    ]

    assert manifest_members, (
        "split_manifest.csv non trovato "
        "nello ZIP YOLO"
    )

    manifest_member = manifest_members[0]

    with archive.open(
        manifest_member
    ) as manifest_file:
        used_manifest = pd.read_csv(
            manifest_file
        )

print("Manifest:", manifest_member)
print("Righe:", len(used_manifest))
print("Colonne:", used_manifest.columns.tolist())

display(
    used_manifest.head(10)
)


In [ ]:
import pandas as pd

all_records = []

for category_directory in sorted(categories):

    category = category_directory.name

    for image_path in category_directory.rglob("*"):

        if (
            not image_path.is_file()
            or image_path.suffix.lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        relative_path = image_path.relative_to(
            category_directory
        )

        assert len(relative_path.parts) >= 2, (
            f"Percorso inatteso: {image_path}"
        )

        group = relative_path.parts[0]

        all_records.append({
            "category": category,
            "group": group,
            "filename": image_path.name,
            "path": str(image_path)
        })


all_images_df = pd.DataFrame(
    all_records
)

used_keys = set(
    zip(
        used_manifest["category"].astype(str),
        used_manifest["group"].astype(str),
        used_manifest["filename"].astype(str)
    )
)

all_images_df["used_for_yolo"] = [
    (
        row.category,
        row.group,
        row.filename
    ) in used_keys
    for row in all_images_df.itertuples()
]

matched_used_df = all_images_df[
    all_images_df["used_for_yolo"]
].copy()

remaining_images_df = all_images_df[
    ~all_images_df["used_for_yolo"]
].copy()

print("Immagini complete:", len(all_images_df))
print("Immagini del manifest trovate:", len(matched_used_df))
print("Immagini rimanenti:", len(remaining_images_df))

assert len(all_images_df) == 12441, (
    f"Totale inatteso: {len(all_images_df)}"
)

assert len(matched_used_df) == 600, (
    "Non sono state trovate esattamente "
    f"600 immagini: trovate {len(matched_used_df)}"
)

assert len(remaining_images_df) == 11841, (
    "Numero rimanente inatteso: "
    f"{len(remaining_images_df)}"
)

display(
    remaining_images_df.groupby(
        "category"
    ).size().rename("remaining_images")
)


In [ ]:
from ultralytics import YOLO

assert WEIGHTS_LOCAL.exists(), f"Checkpoint YOLO non trovato: {WEIGHTS_LOCAL}"
yolo_model = YOLO(str(WEIGHTS_LOCAL))
print("Pesi caricati:", WEIGHTS_LOCAL)
print("Classi del modello:", yolo_model.names)


In [ ]:

from pathlib import Path
import hashlib
import json
import re
import shutil
import time

import numpy as np
import pandas as pd

from PIL import Image



SEED = 42

CONFIDENCE_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.70

KIDNEY_SIZE = 224
MARGIN_RATIO = 0.05

YOLO_BATCH_SIZE = 32
CHUNK_SIZE = 250

REBUILD_FULL_DATASET = True

FULL_DATASET_ROOT = Path(
    "/content/Kidney_Classification_Full"
)

FULL_CROPS_ROOT = (
    FULL_DATASET_ROOT
    / "crops"
)

FULL_RESULTS_DRIVE = FULL_DATASET_RESULTS

FULL_RESULTS_DRIVE.mkdir(
    parents=True,
    exist_ok=True
)

FULL_MANIFEST_PATH = (
    FULL_RESULTS_DRIVE
    / "full_classification_manifest.csv"
)

FULL_EXCLUDED_PATH = (
    FULL_RESULTS_DRIVE
    / "full_excluded_images.csv"
)



if REBUILD_FULL_DATASET:

    assert str(FULL_DATASET_ROOT).startswith(
        "/content/Kidney_Classification_Full"
    )

    if FULL_DATASET_ROOT.exists():
        shutil.rmtree(
            FULL_DATASET_ROOT
        )

FULL_CROPS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CLASS_NAMES_FULL = [
    "Cyst",
    "Normal",
    "Stone",
    "Tumor"
]

SPLIT_NAMES = [
    "train",
    "val",
    "test"
]

for split_name in SPLIT_NAMES:
    for class_name in CLASS_NAMES_FULL:

        (
            FULL_CROPS_ROOT
            / split_name
            / class_name
        ).mkdir(
            parents=True,
            exist_ok=True
        )



used_manifest = used_manifest.copy()

used_manifest["split"] = (
    used_manifest["split"]
    .astype(str)
    .str.lower()
    .replace({
        "validation": "val"
    })
)

assert set(
    used_manifest["split"]
).issubset(
    {"train", "val", "test"}
)



used_manifest["group_key"] = (
    used_manifest["category"].astype(str)
    + "::"
    + used_manifest["group"].astype(str)
)

group_split_counts = (
    used_manifest
    .groupby("group_key")["split"]
    .nunique()
)

assert group_split_counts.max() == 1, (
    "Il manifest contiene gruppi assegnati "
    "a split differenti."
)

locked_group_splits = (
    used_manifest
    .drop_duplicates("group_key")
    .set_index("group_key")["split"]
    .to_dict()
)

print(
    "Gruppi con split già definito:",
    len(locked_group_splits)
)



def deterministic_group_split(
    category,
    group,
    seed=SEED
):
    """
    Assegna uno split stabile in base alla coppia
    categoria-gruppo.

    Circa:
        70% train
        15% validation
        15% test
    """

    group_key = (
        f"{category}::{group}"
    )

    if group_key in locked_group_splits:
        return locked_group_splits[
            group_key
        ]

    text = (
        f"{seed}::{group_key}"
    )

    digest = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    value = (
        int(digest[:16], 16)
        / float(16**16)
    )

    if value < 0.70:
        return "train"

    if value < 0.85:
        return "val"

    return "test"


all_images_with_split = (
    all_images_df.copy()
)

all_images_with_split["group_key"] = (
    all_images_with_split[
        "category"
    ].astype(str)
    + "::"
    + all_images_with_split[
        "group"
    ].astype(str)
)

all_images_with_split["split"] = [
    deterministic_group_split(
        category=row.category,
        group=row.group
    )
    for row
    in all_images_with_split.itertuples()
]



groups_by_split = {
    split_name: set(
        all_images_with_split.loc[
            all_images_with_split[
                "split"
            ] == split_name,
            "group_key"
        ]
    )
    for split_name in SPLIT_NAMES
}

assert groups_by_split["train"].isdisjoint(
    groups_by_split["val"]
)

assert groups_by_split["train"].isdisjoint(
    groups_by_split["test"]
)

assert groups_by_split["val"].isdisjoint(
    groups_by_split["test"]
)

print("Controllo gruppi superato: nessuna sovrapposizione.")



print("\nImmagini prima della detection:")

display(
    pd.crosstab(
        all_images_with_split[
            "category"
        ],
        all_images_with_split[
            "split"
        ],
        margins=True,
        margins_name="Totale"
    )
)

print("\nGruppi per split:")

display(
    all_images_with_split
    .drop_duplicates(
        "group_key"
    )
    .groupby("split")
    .size()
    .rename("groups")
)



def create_full_kidney_panel(
    image,
    box,
    margin_ratio=MARGIN_RATIO
):
    image_width, image_height = (
        image.size
    )

    x1, y1, x2, y2 = map(
        float,
        box
    )

    box_width = x2 - x1
    box_height = y2 - y1

    if (
        box_width <= 0
        or box_height <= 0
    ):
        return None, None

    margin_x = (
        box_width
        * margin_ratio
    )

    margin_y = (
        box_height
        * margin_ratio
    )

    x1 = max(
        0,
        int(
            np.floor(
                x1 - margin_x
            )
        )
    )

    y1 = max(
        0,
        int(
            np.floor(
                y1 - margin_y
            )
        )
    )

    x2 = min(
        image_width,
        int(
            np.ceil(
                x2 + margin_x
            )
        )
    )

    y2 = min(
        image_height,
        int(
            np.ceil(
                y2 + margin_y
            )
        )
    )

    if x2 <= x1 or y2 <= y1:
        return None, None

    kidney_crop = image.crop(
        (x1, y1, x2, y2)
    )

    kidney_crop.thumbnail(
        (
            KIDNEY_SIZE,
            KIDNEY_SIZE
        ),
        Image.Resampling.LANCZOS
    )

    panel = Image.new(
        mode="RGB",
        size=(
            KIDNEY_SIZE,
            KIDNEY_SIZE
        ),
        color=(0, 0, 0)
    )

    paste_x = (
        KIDNEY_SIZE
        - kidney_crop.width
    ) // 2

    paste_y = (
        KIDNEY_SIZE
        - kidney_crop.height
    ) // 2

    panel.paste(
        kidney_crop,
        (
            paste_x,
            paste_y
        )
    )

    coordinates = {
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2
    }

    return panel, coordinates



records = []
excluded_records = []

all_paths = (
    all_images_with_split[
        "path"
    ]
    .astype(str)
    .tolist()
)

metadata_by_path = (
    all_images_with_split
    .set_index("path")[
        [
            "category",
            "group",
            "filename",
            "split"
        ]
    ]
    .to_dict(
        orient="index"
    )
)

start_time = time.time()

for chunk_start in range(
    0,
    len(all_paths),
    CHUNK_SIZE
):

    chunk_paths = all_paths[
        chunk_start:
        chunk_start + CHUNK_SIZE
    ]

    results = yolo_model.predict(
        source=chunk_paths,
        imgsz=640,
        conf=CONFIDENCE_THRESHOLD,
        iou=NMS_IOU_THRESHOLD,
        device=YOLO_DEVICE,
        batch=YOLO_BATCH_SIZE,
        max_det=10,
        verbose=False,
        save=False
    )

    assert len(results) == len(
        chunk_paths
    )

    for original_path, result in zip(
        chunk_paths,
        results
    ):
        metadata = metadata_by_path[
            original_path
        ]

        category = str(
            metadata["category"]
        )

        group = str(
            metadata["group"]
        )

        filename = str(
            metadata["filename"]
        )

        split = str(
            metadata["split"]
        )

        image = Image.open(
            original_path
        ).convert("RGB")

        if (
            result.boxes is None
            or len(result.boxes) == 0
        ):
            excluded_records.append({
                "path": original_path,
                "category": category,
                "group": group,
                "filename": filename,
                "split": split,
                "reason": "no_detection"
            })

            continue

        boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        confidences = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )

        confidence_order = (
            np.argsort(
                confidences
            )[::-1]
        )

        selected_indices = (
            confidence_order[:2]
        )

        selected_boxes = boxes[
            selected_indices
        ]

        selected_confidences = (
            confidences[
                selected_indices
            ]
        )

        horizontal_order = np.argsort(
            (
                selected_boxes[:, 0]
                + selected_boxes[:, 2]
            )
            / 2
        )

        selected_boxes = (
            selected_boxes[
                horizontal_order
            ]
        )

        selected_confidences = (
            selected_confidences[
                horizontal_order
            ]
        )

        kidney_panels = []
        valid_coordinates = []
        valid_confidences = []

        for box, confidence in zip(
            selected_boxes,
            selected_confidences
        ):
            panel, coordinates = (
                create_full_kidney_panel(
                    image=image,
                    box=box
                )
            )

            if panel is None:
                continue

            kidney_panels.append(
                panel
            )

            valid_coordinates.append(
                coordinates
            )

            valid_confidences.append(
                float(confidence)
            )

        if len(kidney_panels) == 0:
            excluded_records.append({
                "path": original_path,
                "category": category,
                "group": group,
                "filename": filename,
                "split": split,
                "reason": "no_valid_crop"
            })

            continue

        real_kidneys_used = len(
            kidney_panels
        )

        if len(kidney_panels) == 1:
            kidney_panels.append(
                Image.new(
                    mode="RGB",
                    size=(
                        KIDNEY_SIZE,
                        KIDNEY_SIZE
                    ),
                    color=(0, 0, 0)
                )
            )

        kidney_panels = (
            kidney_panels[:2]
        )

        combined_image = Image.new(
            mode="RGB",
            size=(
                KIDNEY_SIZE * 2,
                KIDNEY_SIZE
            ),
            color=(0, 0, 0)
        )

        combined_image.paste(
            kidney_panels[0],
            (0, 0)
        )

        combined_image.paste(
            kidney_panels[1],
            (
                KIDNEY_SIZE,
                0
            )
        )

        safe_group = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            group
        )

        output_filename = (
            f"{safe_group}__"
            f"{Path(filename).stem}"
            "_kidneys.jpg"
        )

        destination = (
            FULL_CROPS_ROOT
            / split
            / category
            / output_filename
        )

        combined_image.save(
            destination,
            quality=95
        )

        records.append({
            "original_path": original_path,
            "original_filename": filename,
            "category": category,
            "group": group,
            "group_key": (
                f"{category}::{group}"
            ),
            "split": split,
            "crop_path": str(
                destination
            ),
            "detections_found": int(
                len(boxes)
            ),
            "detections_used": int(
                real_kidneys_used
            ),
            "black_panels": int(
                2 - real_kidneys_used
            ),
            "mean_confidence": float(
                np.mean(
                    valid_confidences
                )
            ),
            "min_confidence": float(
                np.min(
                    valid_confidences
                )
            ),
            "selected_boxes": json.dumps(
                valid_coordinates,
                ensure_ascii=False
            )
        })

    pd.DataFrame(
        records
    ).to_csv(
        FULL_MANIFEST_PATH,
        index=False
    )

    pd.DataFrame(
        excluded_records
    ).to_csv(
        FULL_EXCLUDED_PATH,
        index=False
    )

    processed = min(
        chunk_start + CHUNK_SIZE,
        len(all_paths)
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    print(
        f"Elaborate {processed}/"
        f"{len(all_paths)} immagini | "
        f"crop validi: {len(records)} | "
        f"escluse: {len(excluded_records)} | "
        f"tempo: {elapsed_minutes:.1f} min"
    )



full_classification_manifest = (
    pd.DataFrame(
        records
    )
)

full_excluded_images = (
    pd.DataFrame(
        excluded_records
    )
)

full_classification_manifest.to_csv(
    FULL_MANIFEST_PATH,
    index=False
)

full_excluded_images.to_csv(
    FULL_EXCLUDED_PATH,
    index=False
)



print("\nGENERAZIONE COMPLETATA")
print(
    "Immagini originali:",
    len(all_images_with_split)
)
print(
    "Campioni utilizzabili:",
    len(full_classification_manifest)
)
print(
    "Immagini escluse:",
    len(full_excluded_images)
)

print("\nDistribuzione dei crop:")

full_distribution = pd.crosstab(
    full_classification_manifest[
        "category"
    ],
    full_classification_manifest[
        "split"
    ],
    margins=True,
    margins_name="Totale"
)

display(
    full_distribution
)

manifest_groups = (
    full_classification_manifest
    .groupby("group_key")["split"]
    .nunique()
)

assert manifest_groups.max() == 1, (
    "Data leakage: uno stesso gruppo "
    "compare in più split."
)

assert (
    FULL_CROPS_ROOT
    / "train"
).exists()

assert (
    FULL_CROPS_ROOT
    / "val"
).exists()

assert (
    FULL_CROPS_ROOT
    / "test"
).exists()

print(
    "\nNuovo dataset:",
    FULL_CROPS_ROOT
)


### 5.1 Verifica descrittiva sulle 11.841 immagini non annotate


In [ ]:
from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd

YOLO_FULL_RESULTS = FULL_DATASET_RESULTS
YOLO_FULL_RESULTS.mkdir(parents=True, exist_ok=True)

LOCAL_CHECKPOINT_CSV = Path(
    "/content/full_dataset_yolo_predictions.csv"
)
PREDICTIONS_CSV = (
    YOLO_FULL_RESULTS / "full_dataset_yolo_predictions.csv"
)


records = []

pending_paths = remaining_images_df[
    "path"
].tolist()

metadata_by_path = (
    remaining_images_df
    .set_index("path")[
        ["category", "group", "filename"]
    ]
    .to_dict(orient="index")
)

CHUNK_SIZE = 250
BATCH_SIZE = 32

start_time = time.time()


for chunk_start in range(
    0,
    len(pending_paths),
    CHUNK_SIZE
):

    chunk_paths = pending_paths[
        chunk_start:
        chunk_start + CHUNK_SIZE
    ]

    results = yolo_model.predict(
        source=chunk_paths,
        imgsz=640,
        conf=0.25,
        iou=0.70,
        device=YOLO_DEVICE,
        batch=BATCH_SIZE,
        max_det=10,
        verbose=False,
        save=False
    )

    assert len(results) == len(chunk_paths), (
        f"Input: {len(chunk_paths)}, "
        f"risultati: {len(results)}"
    )

    for original_path, result in zip(
        chunk_paths,
        results
    ):

        image_path = str(original_path)

        metadata = metadata_by_path[
            image_path
        ]

        if (
            result.boxes is None
            or len(result.boxes) == 0
        ):
            confidences = []
            boxes_xyxy = []
            box_areas_ratio = []

        else:
            confidences = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
                .astype(float)
                .tolist()
            )

            boxes_xyxy = (
                result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
                .astype(float)
                .tolist()
            )

            image_height, image_width = (
                result.orig_shape
            )

            image_area = float(
                image_height * image_width
            )

            box_areas_ratio = []

            for x1, y1, x2, y2 in boxes_xyxy:

                box_area = max(
                    0.0,
                    (x2 - x1) * (y2 - y1)
                )

                box_areas_ratio.append(
                    box_area / image_area
                )

        records.append({
            "path": image_path,
            "category": metadata["category"],
            "group": metadata["group"],
            "filename": metadata["filename"],
            "detections": len(boxes_xyxy),
            "max_confidence": (
                max(confidences)
                if confidences
                else np.nan
            ),
            "min_confidence": (
                min(confidences)
                if confidences
                else np.nan
            ),
            "mean_confidence": (
                float(np.mean(confidences))
                if confidences
                else np.nan
            ),
            "confidences": json.dumps(
                confidences
            ),
            "boxes_xyxy": json.dumps(
                boxes_xyxy
            ),
            "box_areas_ratio": json.dumps(
                box_areas_ratio
            )
        })

    checkpoint_df = pd.DataFrame(
        records
    )

    checkpoint_df.to_csv(
        LOCAL_CHECKPOINT_CSV,
        index=False
    )

    shutil.copy2(
        LOCAL_CHECKPOINT_CSV,
        PREDICTIONS_CSV
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    print(
        f"Elaborate {len(checkpoint_df)}/"
        f"{len(remaining_images_df)} | "
        f"tempo: {elapsed_minutes:.1f} min"
    )


print()
print("Inferenza completata")
print("Immagini elaborate:", len(records))
print("Risultati salvati in:")
print(PREDICTIONS_CSV)


In [ ]:

predictions_df = pd.read_csv(
    PREDICTIONS_CSV
)

assert len(predictions_df) == 11841

detection_counts = pd.crosstab(
    predictions_df["category"],
    predictions_df["detections"]
)

detection_percentages = (
    pd.crosstab(
        predictions_df["category"],
        predictions_df["detections"],
        normalize="index"
    )
    * 100
)

print("NUMERO DI IMMAGINI")
display(detection_counts)

print("\nPERCENTUALI PER CATEGORIA")
display(
    detection_percentages.round(2)
)

print("\nRIEPILOGO COMPLESSIVO")

overall_counts = (
    predictions_df["detections"]
    .value_counts()
    .sort_index()
    .rename("images")
    .to_frame()
)

overall_counts["percentage"] = (
    overall_counts["images"]
    / len(predictions_df)
    * 100
)

display(
    overall_counts.round(2)
)

print(
    "Confidence media:",
    round(
        predictions_df[
            "mean_confidence"
        ].mean(),
        4
    )
)

print(
    "Confidence mediana:",
    round(
        predictions_df[
            "mean_confidence"
        ].median(),
        4
    )
)

print(
    "Immagini senza rilevamenti:",
    int(
        (
            predictions_df["detections"] == 0
        ).sum()
    )
)

print(
    "Immagini con più di 2 rilevamenti:",
    int(
        (
            predictions_df["detections"] > 2
        ).sum()
    )
)


In [ ]:
from pathlib import Path
import json
import shutil

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image


AUDIT_DIR = (
    YOLO_FULL_RESULTS
    / "visual_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def select_per_category(
    dataframe,
    number_per_category=4,
    sort_column=None,
    ascending=True
):
    selected_parts = []

    for category in [
        "Cyst",
        "Normal",
        "Stone",
        "Tumor"
    ]:
        category_df = dataframe[
            dataframe["category"] == category
        ].copy()

        if sort_column is not None:
            category_df = category_df.sort_values(
                sort_column,
                ascending=ascending
            )

            selected = category_df.head(
                number_per_category
            )

        else:
            selected = category_df.sample(
                n=min(
                    number_per_category,
                    len(category_df)
                ),
                random_state=42
            )

        selected_parts.append(
            selected
        )

    return pd.concat(
        selected_parts,
        ignore_index=True
    )


def create_detection_gallery(
    dataframe,
    title,
    output_path
):
    columns = 4
    rows = 4

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, 18)
    )

    axes = axes.flatten()

    for axis in axes:
        axis.axis("off")

    for axis, (_, row) in zip(
        axes,
        dataframe.iterrows()
    ):
        image = Image.open(
            row["path"]
        ).convert("RGB")

        axis.imshow(image)

        boxes = json.loads(
            row["boxes_xyxy"]
        )

        confidences = json.loads(
            row["confidences"]
        )

        for box, confidence in zip(
            boxes,
            confidences
        ):
            x1, y1, x2, y2 = box

            rectangle = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor="lime",
                facecolor="none"
            )

            axis.add_patch(
                rectangle
            )

            axis.text(
                x1,
                max(0, y1 - 5),
                f"{confidence:.2f}",
                color="yellow",
                fontsize=8,
                backgroundcolor="black"
            )

        maximum_confidence = (
            row["max_confidence"]
        )

        confidence_text = (
            f"{maximum_confidence:.3f}"
            if pd.notna(maximum_confidence)
            else "nessuna"
        )

        axis.set_title(
            f'{row["category"]} | '
            f'{row["group"]}\n'
            f'{row["filename"]}\n'
            f'box: {row["detections"]} | '
            f'max conf: {confidence_text}',
            fontsize=8
        )

        axis.axis("off")

    figure.suptitle(
        title,
        fontsize=18
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.97]
    )

    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(figure)



zero_cases = select_per_category(
    predictions_df[
        predictions_df["detections"] == 0
    ],
    number_per_category=4
)

create_detection_gallery(
    zero_cases,
    "YOLO – immagini senza rilevamenti",
    AUDIT_DIR / "01_zero_detections.png"
)



one_cases = select_per_category(
    predictions_df[
        predictions_df["detections"] == 1
    ],
    number_per_category=4
)

create_detection_gallery(
    one_cases,
    "YOLO – immagini con un solo rilevamento",
    AUDIT_DIR / "02_one_detection.png"
)



multiple_cases = select_per_category(
    predictions_df[
        predictions_df["detections"] > 2
    ],
    number_per_category=4,
    sort_column="detections",
    ascending=False
)

create_detection_gallery(
    multiple_cases,
    "YOLO – immagini con più di due rilevamenti",
    AUDIT_DIR / "03_more_than_two.png"
)



low_confidence_cases = select_per_category(
    predictions_df[
        predictions_df["detections"] > 0
    ],
    number_per_category=4,
    sort_column="max_confidence",
    ascending=True
)

create_detection_gallery(
    low_confidence_cases,
    "YOLO – rilevamenti con confidence più bassa",
    AUDIT_DIR / "04_low_confidence.png"
)


audit_cases = pd.concat([
    zero_cases.assign(
        audit_reason="zero_detections"
    ),
    one_cases.assign(
        audit_reason="one_detection"
    ),
    multiple_cases.assign(
        audit_reason="more_than_two"
    ),
    low_confidence_cases.assign(
        audit_reason="low_confidence"
    )
], ignore_index=True)

audit_cases.to_csv(
    AUDIT_DIR / "selected_audit_cases.csv",
    index=False
)


archive_path = shutil.make_archive(
    "/content/yolo_visual_audit",
    "zip",
    root_dir=AUDIT_DIR
)

final_archive = (
    YOLO_FULL_RESULTS
    / "yolo_visual_audit.zip"
)

shutil.copy2(
    archive_path,
    final_archive
)

print("Gallerie salvate in:", AUDIT_DIR)
print("ZIP salvato in:", final_archive)


## 6. ResNet50 baseline sull'immagine affiancata 448×224


In [ ]:
from pathlib import Path
import random
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.transforms import InterpolationMode

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)



SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )



CROPS_ROOT = FULL_CROPS_ROOT

assert CROPS_ROOT.exists(), (
    f"Cartella crop non trovata: {CROPS_ROOT}"
)

assert (CROPS_ROOT / "train").exists()
assert (CROPS_ROOT / "val").exists()
assert (CROPS_ROOT / "test").exists()

RESULTS_DRIVE = BASELINE_RESULTS

RESULTS_DRIVE.mkdir(
    parents=True,
    exist_ok=True
)

print("Dataset:", CROPS_ROOT)
print("Risultati:", RESULTS_DRIVE)


In [ ]:
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 448

BATCH_SIZE = 16
NUM_WORKERS = 2

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]



train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_HEIGHT, IMAGE_WIDTH),
        interpolation=InterpolationMode.BILINEAR,
        antialias=True
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.08,
            contrast=0.08
        )
    ], p=0.30),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])



evaluation_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_HEIGHT, IMAGE_WIDTH),
        interpolation=InterpolationMode.BILINEAR,
        antialias=True
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])



train_dataset = datasets.ImageFolder(
    CROPS_ROOT / "train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    CROPS_ROOT / "val",
    transform=evaluation_transform
)

test_dataset = datasets.ImageFolder(
    CROPS_ROOT / "test",
    transform=evaluation_transform
)


assert (
    train_dataset.class_to_idx
    == val_dataset.class_to_idx
    == test_dataset.class_to_idx
), "Mappatura delle classi differente tra gli split"


CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

assert NUM_CLASSES == 4, (
    f"Numero di classi inatteso: {NUM_CLASSES}"
)



generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)



print("\nClassi:", CLASS_NAMES)
print("Mappatura:", train_dataset.class_to_idx)

print("\nNumero di immagini:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print(
    "Totale:",
    len(train_dataset)
    + len(val_dataset)
    + len(test_dataset)
)

print(
    "\nForma input ResNet50:",
    f"{IMAGE_HEIGHT}×{IMAGE_WIDTH}"
)


In [ ]:
from collections import Counter


def dataset_distribution(dataset):
    counts = Counter(
        target
        for _, target in dataset.samples
    )

    return {
        dataset.classes[class_index]: counts[class_index]
        for class_index in range(
            len(dataset.classes)
        )
    }


distribution = pd.DataFrame({
    "train": dataset_distribution(train_dataset),
    "validation": dataset_distribution(val_dataset),
    "test": dataset_distribution(test_dataset)
}).fillna(0).astype(int)

display(distribution)

distribution.to_csv(
    RESULTS_DRIVE / "dataset_distribution.csv"
)


In [ ]:
def denormalize_tensor(tensor):
    mean = torch.tensor(
        IMAGENET_MEAN
    ).view(3, 1, 1)

    std = torch.tensor(
        IMAGENET_STD
    ).view(3, 1, 1)

    image = tensor.cpu() * std + mean

    return image.clamp(0, 1)


images, labels = next(
    iter(train_loader)
)

figure, axes = plt.subplots(
    3,
    4,
    figsize=(18, 9)
)

for axis, image, label in zip(
    axes.flatten(),
    images[:12],
    labels[:12]
):
    display_image = denormalize_tensor(
        image
    )

    display_image = (
        display_image
        .permute(1, 2, 0)
        .numpy()
    )

    axis.imshow(display_image)

    axis.set_title(
        CLASS_NAMES[label.item()]
    )

    axis.axis("off")

plt.tight_layout()
plt.show()

print("Forma batch:", images.shape)


In [ ]:
def calculate_metrics(
    labels,
    predictions
):
    accuracy = accuracy_score(
        labels,
        predictions
    )

    balanced_accuracy = balanced_accuracy_score(
        labels,
        predictions
    )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(macro_f1)
    }


def run_epoch(
    model,
    data_loader,
    criterion,
    optimizer=None
):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0

    all_labels = []
    all_predictions = []

    for images, labels in data_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            is_training
        ):
            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            if is_training:
                loss.backward()
                optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss
        / len(data_loader.dataset)
    )

    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )

    metrics["loss"] = float(epoch_loss)

    return metrics


def fit_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs,
    checkpoint_path,
    patience=6,
    phase_name="training"
):
    history = []

    best_f1 = -1.0
    epochs_without_improvement = 0

    for epoch in range(
        1,
        epochs + 1
    ):
        train_metrics = run_epoch(
            model=model,
            data_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer
        )

        val_metrics = run_epoch(
            model=model,
            data_loader=val_loader,
            criterion=criterion
        )

        scheduler.step(
            val_metrics["macro_f1"]
        )

        learning_rates = [
            group["lr"]
            for group in optimizer.param_groups
        ]

        row = {
            "phase": phase_name,
            "epoch": epoch
        }

        for group_index, learning_rate in enumerate(
            learning_rates
        ):
            row[
                f"learning_rate_group_{group_index}"
            ] = learning_rate

        for key, value in train_metrics.items():
            row[f"train_{key}"] = value

        for key, value in val_metrics.items():
            row[f"val_{key}"] = value

        history.append(row)

        learning_rate_text = ", ".join(
            f"{value:.2e}"
            for value in learning_rates
        )

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss {train_metrics['loss']:.4f} | "
            f"train F1 {train_metrics['macro_f1']:.4f} | "
            f"val loss {val_metrics['loss']:.4f} | "
            f"val acc {val_metrics['accuracy']:.4f} | "
            f"val F1 {val_metrics['macro_f1']:.4f} | "
            f"lr [{learning_rate_text}]"
        )

        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": (
                    model.state_dict()
                ),
                "class_names": CLASS_NAMES,
                "class_to_idx": (
                    train_dataset.class_to_idx
                ),
                "best_val_macro_f1": best_f1,
                "epoch": epoch,
                "phase": phase_name,
                "image_height": IMAGE_HEIGHT,
                "image_width": IMAGE_WIDTH,
                "imagenet_mean": IMAGENET_MEAN,
                "imagenet_std": IMAGENET_STD,
                "crop_strategy": (
                    "two independent YOLO kidney crops"
                )
            }, checkpoint_path)

            print(
                "  Nuovo modello migliore salvato: "
                f"F1={best_f1:.4f}"
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):
            print(
                "Early stopping dopo "
                f"{epoch} epoche"
            )

            break

    return pd.DataFrame(history)


In [ ]:
weights = models.ResNet50_Weights.DEFAULT

model = models.resnet50(
    weights=weights
)

for parameter in model.parameters():
    parameter.requires_grad = False

input_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(
        input_features,
        NUM_CLASSES
    )
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Parametri totali:", total_parameters)
print(
    "Parametri addestrabili:",
    trainable_parameters
)
print(model.fc)


In [ ]:
PHASE1_CHECKPOINT = (
    RESULTS_DRIVE
    / "resnet50_yolo_separate_phase1_best.pt"
)

optimizer_phase1 = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler_phase1 = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_phase1,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
)

history_phase1 = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_phase1,
    scheduler=scheduler_phase1,
    epochs=20,
    checkpoint_path=PHASE1_CHECKPOINT,
    patience=6,
    phase_name="frozen_backbone"
)

history_phase1.to_csv(
    RESULTS_DRIVE
    / "history_phase1.csv",
    index=False
)

print("\nFase 1 completata")
print("Checkpoint:", PHASE1_CHECKPOINT)


In [ ]:
phase1_checkpoint = torch.load(
    PHASE1_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    phase1_checkpoint[
        "model_state_dict"
    ]
)

for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.layer4.parameters():
    parameter.requires_grad = True

for parameter in model.fc.parameters():
    parameter.requires_grad = True


PHASE2_CHECKPOINT = (
    RESULTS_DRIVE
    / "resnet50_yolo_separate_phase2_best.pt"
)

optimizer_phase2 = torch.optim.AdamW([
    {
        "params": model.layer4.parameters(),
        "lr": 1e-5
    },
    {
        "params": model.fc.parameters(),
        "lr": 1e-4
    }
], weight_decay=1e-4)

scheduler_phase2 = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_phase2,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7
    )
)

history_phase2 = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_phase2,
    scheduler=scheduler_phase2,
    epochs=25,
    checkpoint_path=PHASE2_CHECKPOINT,
    patience=7,
    phase_name="layer4_finetuning"
)

history_phase2.to_csv(
    RESULTS_DRIVE
    / "history_phase2.csv",
    index=False
)

print("\nFase 2 completata")
print("Checkpoint:", PHASE2_CHECKPOINT)


In [ ]:
best_phase1 = (
    history_phase1
    .sort_values(
        by="val_macro_f1",
        ascending=False
    )
    .iloc[0]
)

best_phase2 = (
    history_phase2
    .sort_values(
        by="val_macro_f1",
        ascending=False
    )
    .iloc[0]
)

comparison = pd.DataFrame([
    {
        "phase": "Fase 1 - testa",
        "best_epoch": int(
            best_phase1["epoch"]
        ),
        "val_loss": float(
            best_phase1["val_loss"]
        ),
        "val_accuracy": float(
            best_phase1["val_accuracy"]
        ),
        "val_balanced_accuracy": float(
            best_phase1[
                "val_balanced_accuracy"
            ]
        ),
        "val_macro_f1": float(
            best_phase1["val_macro_f1"]
        )
    },
    {
        "phase": "Fase 2 - layer4",
        "best_epoch": int(
            best_phase2["epoch"]
        ),
        "val_loss": float(
            best_phase2["val_loss"]
        ),
        "val_accuracy": float(
            best_phase2["val_accuracy"]
        ),
        "val_balanced_accuracy": float(
            best_phase2[
                "val_balanced_accuracy"
            ]
        ),
        "val_macro_f1": float(
            best_phase2["val_macro_f1"]
        )
    }
])

display(comparison.round(4))

comparison.to_csv(
    RESULTS_DRIVE
    / "validation_comparison.csv",
    index=False
)


In [ ]:
def plot_training_curves(phase1_path, phase2_path, output_path, title):
    phase1 = pd.read_csv(phase1_path).copy()
    phase2 = pd.read_csv(phase2_path).copy()
    phase1["global_epoch"] = phase1["epoch"]
    phase2["global_epoch"] = phase2["epoch"] + phase1["epoch"].max()
    history = pd.concat([phase1, phase2], ignore_index=True)
    boundary = phase1["epoch"].max() + 0.5

    best1 = phase1.loc[phase1["val_macro_f1"].idxmax()]
    best2 = phase2.loc[phase2["val_macro_f1"].idxmax()]

    figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    axes[0].plot(history["global_epoch"], history["train_loss"], label="Training")
    axes[0].plot(history["global_epoch"], history["val_loss"], label="Validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoca")
    axes[0].legend()

    axes[1].plot(history["global_epoch"], history["train_macro_f1"], label="Training")
    axes[1].plot(history["global_epoch"], history["val_macro_f1"], label="Validation")
    axes[1].scatter(best1["epoch"], best1["val_macro_f1"], marker="*", s=160, color="black")
    axes[1].scatter(
        best2["epoch"] + phase1["epoch"].max(),
        best2["val_macro_f1"],
        marker="*",
        s=160,
        color="black",
    )
    axes[1].set_title("Macro-F1")
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    for axis in axes:
        axis.axvline(boundary, color="gray", linestyle="--", label="Fine fase 1")

    figure.suptitle(title, fontsize=15, fontweight="bold")
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return best1, best2


In [ ]:
baseline_best_phase1, baseline_best_phase2 = plot_training_curves(
    RESULTS_DRIVE / "history_phase1.csv",
    RESULTS_DRIVE / "history_phase2.csv",
    RESULTS_DRIVE / "baseline_training_curves.png",
    "Addestramento ResNet50 baseline",
)
print("Baseline, miglior Macro-F1 validation:", round(float(baseline_best_phase2["val_macro_f1"]), 4))


In [ ]:
print("Campioni nel test dataset:", len(test_dataset))
print("Campioni nel test loader:", len(test_loader.dataset))

assert test_loader.dataset is test_dataset
assert len(test_dataset) == 1917, (
    "ERRORE: il test finale della baseline deve contenere "
    "esattamente 1.917 immagini."
)


In [ ]:
import json
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

best_checkpoint = torch.load(
    PHASE2_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

test_labels = []
test_predictions = []
test_probabilities = []

running_test_loss = 0.0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = criterion(logits, labels)

        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        running_test_loss += (
            loss.item() * images.size(0)
        )

        test_labels.extend(
            labels.cpu().numpy()
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

test_labels = np.asarray(test_labels)
test_predictions = np.asarray(test_predictions)
test_probabilities = np.asarray(test_probabilities)

test_loss = running_test_loss / len(test_dataset)

test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)

test_balanced_accuracy = balanced_accuracy_score(
    test_labels,
    test_predictions
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        test_labels,
        test_predictions,
        average="macro",
        zero_division=0
    )
)

test_macro_auc = roc_auc_score(
    test_labels,
    test_probabilities,
    multi_class="ovr",
    average="macro"
)

test_weighted_auc = roc_auc_score(
    test_labels,
    test_probabilities,
    multi_class="ovr",
    average="weighted"
)

final_metrics = {
    "checkpoint": str(PHASE2_CHECKPOINT),
    "selected_on": "validation_macro_f1",
    "checkpoint_epoch": int(
        best_checkpoint["epoch"]
    ),
    "best_validation_macro_f1": float(
        best_checkpoint["best_val_macro_f1"]
    ),
    "test_samples": int(len(test_dataset)),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_balanced_accuracy": float(
        test_balanced_accuracy
    ),
    "test_macro_precision": float(macro_precision),
    "test_macro_recall": float(macro_recall),
    "test_macro_f1": float(macro_f1),
    "test_macro_roc_auc_ovr": float(test_macro_auc),
    "test_weighted_roc_auc_ovr": float(
        test_weighted_auc
    )
}

print("RISULTATI FINALI SUL TEST")
print("=" * 45)

for key, value in final_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

with open(
    RESULTS_DRIVE / "test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metrics,
        file,
        indent=2,
        ensure_ascii=False
    )


In [ ]:
report = classification_report(
    test_labels,
    test_predictions,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

display(
    report_df.round(4)
)

report_df.to_csv(
    RESULTS_DRIVE / "test_classification_report.csv"
)

confusion = confusion_matrix(
    test_labels,
    test_predictions
)

fig, axis = plt.subplots(figsize=(8, 7))

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=CLASS_NAMES
)

display_cm.plot(
    ax=axis,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

axis.set_title(
    "ResNet50 – Confusion matrix sul test"
)

plt.tight_layout()

plt.savefig(
    RESULTS_DRIVE / "test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    auc
)

fig, axis = plt.subplots(figsize=(9, 7))

roc_rows = []

for class_index, class_name in enumerate(CLASS_NAMES):
    binary_labels = (
        test_labels == class_index
    ).astype(int)

    false_positive_rate, true_positive_rate, _ = roc_curve(
        binary_labels,
        test_probabilities[:, class_index]
    )

    class_auc = auc(
        false_positive_rate,
        true_positive_rate
    )

    axis.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"{class_name} (AUC={class_auc:.3f})"
    )

    for fpr_value, tpr_value in zip(
        false_positive_rate,
        true_positive_rate
    ):
        roc_rows.append({
            "class": class_name,
            "fpr": fpr_value,
            "tpr": tpr_value,
            "auc": class_auc
        })

axis.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Classificatore casuale"
)

axis.set_xlabel("False Positive Rate")
axis.set_ylabel("True Positive Rate")
axis.set_title("ResNet50 – ROC one-vs-rest sul test")
axis.legend()
axis.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    RESULTS_DRIVE / "test_roc_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

pd.DataFrame(roc_rows).to_csv(
    RESULTS_DRIVE / "test_roc_curves.csv",
    index=False
)


In [ ]:
import numpy as np
import pandas as pd
import torch

is_random_sampler = isinstance(
    test_loader.sampler,
    torch.utils.data.RandomSampler
)

print("Test loader shuffle:", is_random_sampler)

if is_random_sampler:
    raise RuntimeError(
        "test_loader usa shuffle=True: percorsi e predizioni "
        "potrebbero non corrispondere."
    )

test_paths = [
    path
    for path, _ in test_dataset.samples
]

test_labels = np.asarray(test_labels)
test_predictions = np.asarray(test_predictions)
test_probabilities = np.asarray(test_probabilities)

assert len(test_paths) == len(test_labels), (
    f"Percorsi: {len(test_paths)}, etichette: {len(test_labels)}"
)

assert len(test_paths) == len(test_predictions), (
    f"Percorsi: {len(test_paths)}, predizioni: "
    f"{len(test_predictions)}"
)

assert len(test_paths) == len(test_probabilities), (
    f"Percorsi: {len(test_paths)}, probabilità: "
    f"{len(test_probabilities)}"
)

error_analysis = pd.DataFrame({
    "path": test_paths,
    "true_index": test_labels.astype(int),
    "predicted_index": test_predictions.astype(int),
    "true_class": [
        CLASS_NAMES[int(index)]
        for index in test_labels
    ],
    "predicted_class": [
        CLASS_NAMES[int(index)]
        for index in test_predictions
    ],
    "confidence": test_probabilities.max(axis=1)
})

error_analysis["correct"] = (
    error_analysis["true_index"]
    == error_analysis["predicted_index"]
)

for class_index, class_name in enumerate(CLASS_NAMES):
    error_analysis[f"probability_{class_name}"] = (
        test_probabilities[:, class_index]
    )

errors = (
    error_analysis.loc[
        ~error_analysis["correct"]
    ]
    .sort_values(
        by="confidence",
        ascending=False
    )
    .reset_index(drop=True)
)

display(errors)

error_analysis.to_csv(
    RESULTS_DRIVE / "test_predictions.csv",
    index=False
)

errors.to_csv(
    RESULTS_DRIVE / "test_errors.csv",
    index=False
)

print()
print("Immagini totali:", len(error_analysis))
print("Corrette:", int(error_analysis["correct"].sum()))
print("Errate:", len(errors))


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


with open(
    RESULTS_DRIVE / "test_metrics.json",
    "r",
    encoding="utf-8"
) as file:
    test_metrics_data = json.load(file)

classification_report_df = pd.read_csv(
    RESULTS_DRIVE / "test_classification_report.csv",
    index_col=0
)

class_report = classification_report_df.loc[
    CLASS_NAMES
].copy()


overall_metric_names = [
    "Accuracy",
    "Balanced accuracy",
    "Macro precision",
    "Macro recall",
    "Macro F1",
    "Macro ROC-AUC"
]

overall_metric_values = [
    test_metrics_data["test_accuracy"],
    test_metrics_data["test_balanced_accuracy"],
    test_metrics_data["test_macro_precision"],
    test_metrics_data["test_macro_recall"],
    test_metrics_data["test_macro_f1"],
    test_metrics_data["test_macro_roc_auc_ovr"]
]

best_validation_f1 = test_metrics_data[
    "best_validation_macro_f1"
]

test_macro_f1 = test_metrics_data[
    "test_macro_f1"
]


plt.style.use("seaborn-v0_8-whitegrid")

fig, axes = plt.subplots(
    1,
    3,
    figsize=(19, 6),
    constrained_layout=True
)


ax = axes[0]

colors = [
    "#4C78A8",
    "#72B7B2",
    "#F58518",
    "#E45756",
    "#54A24B",
    "#B279A2"
]

bars = ax.bar(
    overall_metric_names,
    overall_metric_values,
    color=colors
)

ax.set_title("Metriche complessive sul test set")
ax.set_ylabel("Valore")
ax.set_ylim(0, 1.08)
ax.tick_params(axis="x", rotation=45)

ax.bar_label(
    bars,
    labels=[
        f"{value:.3f}"
        for value in overall_metric_values
    ],
    padding=3,
    fontsize=9
)


ax = axes[1]

x = np.arange(len(CLASS_NAMES))
bar_width = 0.25

precision_bars = ax.bar(
    x - bar_width,
    class_report["precision"],
    width=bar_width,
    label="Precision",
    color="#4C78A8"
)

recall_bars = ax.bar(
    x,
    class_report["recall"],
    width=bar_width,
    label="Recall",
    color="#F58518"
)

f1_bars = ax.bar(
    x + bar_width,
    class_report["f1-score"],
    width=bar_width,
    label="F1-score",
    color="#54A24B"
)

ax.set_title("Metriche per classe sul test set")
ax.set_ylabel("Valore")
ax.set_ylim(0, 1.08)
ax.set_xticks(x)
ax.set_xticklabels(
    CLASS_NAMES,
    rotation=30,
    ha="right"
)
ax.legend()

for bar_group in [
    precision_bars,
    recall_bars,
    f1_bars
]:
    ax.bar_label(
        bar_group,
        fmt="%.2f",
        padding=2,
        fontsize=8,
        rotation=90
    )


ax = axes[2]

comparison_names = [
    "Migliore validation\nMacro F1",
    "Test\nMacro F1"
]

comparison_values = [
    best_validation_f1,
    test_macro_f1
]

comparison_bars = ax.bar(
    comparison_names,
    comparison_values,
    color=["#72B7B2", "#E45756"],
    width=0.6
)

ax.set_title("Confronto validation–test")
ax.set_ylabel("Macro F1-score")
ax.set_ylim(0, 1.08)

ax.bar_label(
    comparison_bars,
    labels=[
        f"{value:.4f}"
        for value in comparison_values
    ],
    padding=4,
    fontsize=10
)

difference = test_macro_f1 - best_validation_f1

ax.text(
    0.5,
    0.05,
    f"Differenza test − validation: {difference:+.4f}",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    bbox={
        "boxstyle": "round,pad=0.4",
        "facecolor": "white",
        "edgecolor": "gray"
    }
)

fig.suptitle(
    "Valutazione finale di ResNet50 sul test set",
    fontsize=16,
    fontweight="bold"
)


OUTPUT_PNG = (
    RESULTS_DRIVE
    / "resnet50_test_metrics_summary.png"
)

OUTPUT_PDF = (
    RESULTS_DRIVE
    / "resnet50_test_metrics_summary.pdf"
)

fig.savefig(
    OUTPUT_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    OUTPUT_PDF,
    bbox_inches="tight"
)

plt.show()

print("Grafico PNG salvato in:", OUTPUT_PNG)
print("Grafico PDF salvato in:", OUTPUT_PDF)


## 7. ResNet50 a due rami con masked max pooling


In [ ]:
from pathlib import Path
from PIL import Image

import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode


KIDNEY_IMAGE_SIZE = 224
BLACK_THRESHOLD = 8


def remove_black_padding(
    image,
    threshold=BLACK_THRESHOLD
):
    """
    Rimuove il padding nero esterno dal singolo pannello.
    """

    array = np.asarray(
        image.convert("RGB")
    )

    non_black_mask = (
        array.max(axis=2) > threshold
    )

    rows, columns = np.where(
        non_black_mask
    )

    if len(rows) == 0:
        return None

    x1 = int(columns.min())
    x2 = int(columns.max()) + 1

    y1 = int(rows.min())
    y2 = int(rows.max()) + 1

    return image.crop(
        (x1, y1, x2, y2)
    )


class TwoKidneyDataset(Dataset):

    def __init__(
        self,
        root_directory,
        transform=None
    ):
        self.base_dataset = datasets.ImageFolder(
            root_directory
        )

        self.samples = self.base_dataset.samples
        self.classes = self.base_dataset.classes
        self.class_to_idx = (
            self.base_dataset.class_to_idx
        )

        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        combined_image = Image.open(
            image_path
        ).convert("RGB")

        width, height = combined_image.size
        middle = width // 2

        left_panel = combined_image.crop(
            (0, 0, middle, height)
        )

        right_panel = combined_image.crop(
            (middle, 0, width, height)
        )

        left_crop = remove_black_padding(
            left_panel
        )

        right_crop = remove_black_padding(
            right_panel
        )

        left_valid = left_crop is not None
        right_valid = right_crop is not None

        if not left_valid and not right_valid:
            raise RuntimeError(
                "Entrambi i pannelli sono vuoti: "
                f"{image_path}"
            )

        if not left_valid:
            left_crop = right_crop.copy()

        if not right_valid:
            right_crop = left_crop.copy()

        if self.transform is not None:
            left_tensor = self.transform(
                left_crop
            )

            right_tensor = self.transform(
                right_crop
            )

        valid_mask = torch.tensor(
            [left_valid, right_valid],
            dtype=torch.bool
        )

        return (
            left_tensor,
            right_tensor,
            valid_mask,
            label
        )


In [ ]:
IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

BATCH_SIZE = 16
NUM_WORKERS = 2


train_transform_two_branch = transforms.Compose([
    transforms.Resize(
        (
            KIDNEY_IMAGE_SIZE,
            KIDNEY_IMAGE_SIZE
        ),
        interpolation=InterpolationMode.BILINEAR,
        antialias=True
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.08,
            contrast=0.08
        )
    ], p=0.30),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


evaluation_transform_two_branch = transforms.Compose([
    transforms.Resize(
        (
            KIDNEY_IMAGE_SIZE,
            KIDNEY_IMAGE_SIZE
        ),
        interpolation=InterpolationMode.BILINEAR,
        antialias=True
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


train_dataset_two = TwoKidneyDataset(
    CROPS_ROOT / "train",
    transform=train_transform_two_branch
)

val_dataset_two = TwoKidneyDataset(
    CROPS_ROOT / "val",
    transform=evaluation_transform_two_branch
)

test_dataset_two = TwoKidneyDataset(
    CROPS_ROOT / "test",
    transform=evaluation_transform_two_branch
)


assert (
    train_dataset_two.class_to_idx
    == val_dataset_two.class_to_idx
    == test_dataset_two.class_to_idx
)


CLASS_NAMES = train_dataset_two.classes
NUM_CLASSES = len(CLASS_NAMES)

generator = torch.Generator()
generator.manual_seed(SEED)


train_loader_two = DataLoader(
    train_dataset_two,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    generator=generator
)

val_loader_two = DataLoader(
    val_dataset_two,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)

test_loader_two = DataLoader(
    test_dataset_two,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)


print("Classi:", CLASS_NAMES)
print("Mappatura:", train_dataset_two.class_to_idx)

print("Train:", len(train_dataset_two))
print("Validation:", len(val_dataset_two))
print("Test:", len(test_dataset_two))


In [ ]:
def denormalize_kidney(tensor):
    mean = torch.tensor(
        IMAGENET_MEAN
    ).view(3, 1, 1)

    std = torch.tensor(
        IMAGENET_STD
    ).view(3, 1, 1)

    image = (
        tensor.cpu() * std + mean
    )

    return image.clamp(0, 1)


left_images, right_images, masks, labels = (
    next(iter(train_loader_two))
)

figure, axes = plt.subplots(
    4,
    4,
    figsize=(12, 12)
)

for sample_index in range(8):

    left_image = denormalize_kidney(
        left_images[sample_index]
    ).permute(1, 2, 0).numpy()

    right_image = denormalize_kidney(
        right_images[sample_index]
    ).permute(1, 2, 0).numpy()

    axes[
        sample_index // 2,
        (sample_index % 2) * 2
    ].imshow(left_image)

    axes[
        sample_index // 2,
        (sample_index % 2) * 2
    ].set_title(
        f"{CLASS_NAMES[labels[sample_index]]} – rene 1\n"
        f"valido: {bool(masks[sample_index, 0])}"
    )

    axes[
        sample_index // 2,
        (sample_index % 2) * 2 + 1
    ].imshow(right_image)

    axes[
        sample_index // 2,
        (sample_index % 2) * 2 + 1
    ].set_title(
        f"{CLASS_NAMES[labels[sample_index]]} – rene 2\n"
        f"valido: {bool(masks[sample_index, 1])}"
    )

for axis in axes.flatten():
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch.nn as nn

from torchvision import models


class TwoBranchResNet50(nn.Module):

    def __init__(
        self,
        number_of_classes,
        dropout=0.35
    ):
        super().__init__()

        weights = (
            models.ResNet50_Weights.DEFAULT
        )

        self.encoder = models.resnet50(
            weights=weights
        )

        feature_size = (
            self.encoder.fc.in_features
        )

        self.encoder.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(
                feature_size,
                number_of_classes
            )
        )

    def forward(
        self,
        left_image,
        right_image,
        valid_mask
    ):
        batch_size = left_image.size(0)

        combined_batch = torch.cat(
            [left_image, right_image],
            dim=0
        )

        combined_features = self.encoder(
            combined_batch
        )

        left_features = combined_features[
            :batch_size
        ]

        right_features = combined_features[
            batch_size:
        ]

        features = torch.stack(
            [
                left_features,
                right_features
            ],
            dim=1
        )

        valid_mask = valid_mask.to(
            features.device
        )

        masked_features = features.masked_fill(
            ~valid_mask.unsqueeze(-1),
            float("-inf")
        )

        fused_features = masked_features.max(
            dim=1
        ).values

        logits = self.classifier(
            fused_features
        )

        return logits


model_two = TwoBranchResNet50(
    number_of_classes=NUM_CLASSES,
    dropout=0.35
).to(DEVICE)

criterion_two = nn.CrossEntropyLoss()

print(model_two.classifier)


In [ ]:
def run_epoch_two_branch(
    model,
    data_loader,
    criterion,
    optimizer=None
):
    is_training = optimizer is not None

    if is_training:
        model.train()

        model.encoder.eval()

        if any(
            parameter.requires_grad
            for parameter
            in model.encoder.layer4.parameters()
        ):
            model.encoder.layer4.train()

        model.classifier.train()

    else:
        model.eval()

    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for (
        left_images,
        right_images,
        valid_masks,
        labels
    ) in data_loader:

        left_images = left_images.to(
            DEVICE,
            non_blocking=True
        )

        right_images = right_images.to(
            DEVICE,
            non_blocking=True
        )

        valid_masks = valid_masks.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            is_training
        ):
            logits = model(
                left_images,
                right_images,
                valid_masks
            )

            loss = criterion(
                logits,
                labels
            )

            if is_training:
                loss.backward()
                optimizer.step()

        running_loss += (
            loss.item()
            * labels.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss
        / len(data_loader.dataset)
    )

    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )

    metrics["loss"] = float(
        epoch_loss
    )

    return metrics


def fit_two_branch(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs,
    checkpoint_path,
    patience,
    phase_name
):
    history = []

    best_f1 = -1.0
    epochs_without_improvement = 0

    for epoch in range(
        1,
        epochs + 1
    ):
        train_metrics = (
            run_epoch_two_branch(
                model=model,
                data_loader=train_loader,
                criterion=criterion,
                optimizer=optimizer
            )
        )

        val_metrics = (
            run_epoch_two_branch(
                model=model,
                data_loader=val_loader,
                criterion=criterion
            )
        )

        scheduler.step(
            val_metrics["macro_f1"]
        )

        learning_rates = [
            group["lr"]
            for group in optimizer.param_groups
        ]

        row = {
            "phase": phase_name,
            "epoch": epoch
        }

        for index, learning_rate in enumerate(
            learning_rates
        ):
            row[
                f"learning_rate_group_{index}"
            ] = learning_rate

        for key, value in train_metrics.items():
            row[f"train_{key}"] = value

        for key, value in val_metrics.items():
            row[f"val_{key}"] = value

        history.append(row)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss {train_metrics['loss']:.4f} | "
            f"train F1 {train_metrics['macro_f1']:.4f} | "
            f"val loss {val_metrics['loss']:.4f} | "
            f"val acc {val_metrics['accuracy']:.4f} | "
            f"val F1 {val_metrics['macro_f1']:.4f} | "
            f"lr {learning_rates}"
        )

        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": (
                    model.state_dict()
                ),
                "class_names": CLASS_NAMES,
                "class_to_idx": (
                    train_dataset_two.class_to_idx
                ),
                "best_val_macro_f1": best_f1,
                "epoch": epoch,
                "phase": phase_name,
                "kidney_image_size": (
                    KIDNEY_IMAGE_SIZE
                ),
                "fusion": "masked_max",
                "padding_removed": True
            }, checkpoint_path)

            print(
                "  Nuovo modello migliore: "
                f"F1={best_f1:.4f}"
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):
            print(
                f"Early stopping dopo "
                f"{epoch} epoche"
            )
            break

    return pd.DataFrame(history)


In [ ]:
TWO_BRANCH_RESULTS = TWO_BRANCH_RESULTS_ROOT

TWO_BRANCH_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)

for parameter in model_two.encoder.parameters():
    parameter.requires_grad = False

for parameter in model_two.classifier.parameters():
    parameter.requires_grad = True


TWO_PHASE1_CHECKPOINT = (
    TWO_BRANCH_RESULTS
    / "two_branch_phase1_best.pt"
)

optimizer_two_phase1 = torch.optim.AdamW(
    model_two.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler_two_phase1 = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_two_phase1,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
)

history_two_phase1 = fit_two_branch(
    model=model_two,
    train_loader=train_loader_two,
    val_loader=val_loader_two,
    criterion=criterion_two,
    optimizer=optimizer_two_phase1,
    scheduler=scheduler_two_phase1,
    epochs=20,
    checkpoint_path=TWO_PHASE1_CHECKPOINT,
    patience=6,
    phase_name="two_branch_frozen"
)

history_two_phase1.to_csv(
    TWO_BRANCH_RESULTS
    / "history_phase1.csv",
    index=False
)


In [ ]:
phase1_checkpoint = torch.load(
    TWO_PHASE1_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model_two.load_state_dict(
    phase1_checkpoint[
        "model_state_dict"
    ]
)

for parameter in model_two.parameters():
    parameter.requires_grad = False

for parameter in model_two.encoder.layer4.parameters():
    parameter.requires_grad = True

for parameter in model_two.classifier.parameters():
    parameter.requires_grad = True


TWO_PHASE2_CHECKPOINT = (
    TWO_BRANCH_RESULTS
    / "two_branch_phase2_best.pt"
)

optimizer_two_phase2 = torch.optim.AdamW([
    {
        "params": (
            model_two.encoder.layer4.parameters()
        ),
        "lr": 1e-5
    },
    {
        "params": (
            model_two.classifier.parameters()
        ),
        "lr": 1e-4
    }
], weight_decay=1e-4)

scheduler_two_phase2 = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_two_phase2,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7
    )
)

history_two_phase2 = fit_two_branch(
    model=model_two,
    train_loader=train_loader_two,
    val_loader=val_loader_two,
    criterion=criterion_two,
    optimizer=optimizer_two_phase2,
    scheduler=scheduler_two_phase2,
    epochs=25,
    checkpoint_path=TWO_PHASE2_CHECKPOINT,
    patience=7,
    phase_name="two_branch_layer4"
)

history_two_phase2.to_csv(
    TWO_BRANCH_RESULTS
    / "history_phase2.csv",
    index=False
)


In [ ]:
two_branch_best_phase1, two_branch_best_phase2 = plot_training_curves(
    TWO_BRANCH_RESULTS / "history_phase1.csv",
    TWO_BRANCH_RESULTS / "history_phase2.csv",
    TWO_BRANCH_RESULTS / "two_branch_training_curves.png",
    "Addestramento ResNet50 a due rami",
)
print("Due rami, miglior Macro-F1 validation:", round(float(two_branch_best_phase2["val_macro_f1"]), 4))


### 7.1 Valutazione finale sul test set da 1.917 immagini


In [ ]:
import json
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

print(
    "Campioni nel test dataset a due rami:",
    len(test_loader_two.dataset)
)

assert len(test_loader_two.dataset) == 1917, (
    "ERRORE: il test finale del modello a due rami deve "
    "contenere esattamente 1.917 immagini."
)

best_two_checkpoint = torch.load(
    TWO_PHASE2_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model_two.load_state_dict(
    best_two_checkpoint["model_state_dict"]
)

model_two = model_two.to(DEVICE)
model_two.eval()

two_test_labels = []
two_test_predictions = []
two_test_probabilities = []

running_two_test_loss = 0.0

with torch.no_grad():

    for (
        left_images,
        right_images,
        valid_mask,
        labels
    ) in test_loader_two:

        left_images = left_images.to(
            DEVICE,
            non_blocking=True
        )

        right_images = right_images.to(
            DEVICE,
            non_blocking=True
        )

        valid_mask = valid_mask.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        logits = model_two(
            left_images,
            right_images,
            valid_mask
        )

        loss = criterion_two(
            logits,
            labels
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = probabilities.argmax(
            dim=1
        )

        running_two_test_loss += (
            loss.item() * labels.size(0)
        )

        two_test_labels.extend(
            labels.cpu().numpy()
        )

        two_test_predictions.extend(
            predictions.cpu().numpy()
        )

        two_test_probabilities.extend(
            probabilities.cpu().numpy()
        )

two_test_labels = np.asarray(
    two_test_labels
)

two_test_predictions = np.asarray(
    two_test_predictions
)

two_test_probabilities = np.asarray(
    two_test_probabilities
)

unique_classes, class_counts = np.unique(
    two_test_labels,
    return_counts=True
)

print(
    "Classi presenti nel test:",
    dict(zip(unique_classes, class_counts))
)

two_test_loss = (
    running_two_test_loss
    / len(test_loader_two.dataset)
)

two_test_accuracy = accuracy_score(
    two_test_labels,
    two_test_predictions
)

two_test_balanced_accuracy = (
    balanced_accuracy_score(
        two_test_labels,
        two_test_predictions
    )
)

(
    two_macro_precision,
    two_macro_recall,
    two_macro_f1,
    _
) = precision_recall_fscore_support(
    two_test_labels,
    two_test_predictions,
    average="macro",
    zero_division=0
)

two_test_macro_auc = roc_auc_score(
    two_test_labels,
    two_test_probabilities,
    multi_class="ovr",
    average="macro"
)

two_test_weighted_auc = roc_auc_score(
    two_test_labels,
    two_test_probabilities,
    multi_class="ovr",
    average="weighted"
)

two_final_metrics = {
    "checkpoint": str(TWO_PHASE2_CHECKPOINT),
    "selected_on": "validation_macro_f1",
    "checkpoint_epoch": int(
        best_two_checkpoint["epoch"]
    ),
    "best_validation_macro_f1": float(
        best_two_checkpoint[
            "best_val_macro_f1"
        ]
    ),
    "test_samples": int(
        len(test_loader_two.dataset)
    ),
    "test_loss": float(two_test_loss),
    "test_accuracy": float(
        two_test_accuracy
    ),
    "test_balanced_accuracy": float(
        two_test_balanced_accuracy
    ),
    "test_macro_precision": float(
        two_macro_precision
    ),
    "test_macro_recall": float(
        two_macro_recall
    ),
    "test_macro_f1": float(
        two_macro_f1
    ),
    "test_macro_roc_auc_ovr": float(
        two_test_macro_auc
    ),
    "test_weighted_roc_auc_ovr": float(
        two_test_weighted_auc
    )
}

print("\nRISULTATI FINALI — MODELLO A DUE RAMI")
print("=" * 55)

for key, value in two_final_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

with open(
    TWO_BRANCH_RESULTS
    / "test_metrics_two_branch.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        two_final_metrics,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    "\nMetriche salvate in:",
    TWO_BRANCH_RESULTS
    / "test_metrics_two_branch.json"
)


In [ ]:
two_test_labels = []
two_test_predictions = []
two_test_probabilities = []

model_two.eval()

with torch.no_grad():

    for (
        left_images,
        right_images,
        valid_masks,
        labels
    ) in test_loader_two:

        left_images = left_images.to(DEVICE)
        right_images = right_images.to(DEVICE)
        valid_masks = valid_masks.to(DEVICE)

        logits = model_two(
            left_images,
            right_images,
            valid_masks
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = probabilities.argmax(
            dim=1
        )

        two_test_labels.extend(
            labels.numpy()
        )

        two_test_predictions.extend(
            predictions.cpu().numpy()
        )

        two_test_probabilities.extend(
            probabilities.cpu().numpy()
        )


two_test_labels = np.asarray(
    two_test_labels
)

two_test_predictions = np.asarray(
    two_test_predictions
)

two_test_probabilities = np.asarray(
    two_test_probabilities
)

two_test_paths = [
    path
    for path, _ in test_dataset_two.samples
]

two_gradcam_results = pd.DataFrame({
    "path": two_test_paths,
    "true_index": two_test_labels.astype(int),
    "predicted_index": (
        two_test_predictions.astype(int)
    ),
    "true_class": [
        CLASS_NAMES[index]
        for index in two_test_labels
    ],
    "predicted_class": [
        CLASS_NAMES[index]
        for index in two_test_predictions
    ],
    "confidence": (
        two_test_probabilities.max(axis=1)
    )
})

two_gradcam_results["correct"] = (
    two_gradcam_results["true_index"]
    == two_gradcam_results["predicted_index"]
)

two_gradcam_results.to_csv(
    TWO_BRANCH_RESULTS / "test_predictions.csv",
    index=False
)

for class_index, class_name in enumerate(
    CLASS_NAMES
):
    two_gradcam_results[
        f"probability_{class_name}"
    ] = two_test_probabilities[
        :,
        class_index
    ]

two_errors = (
    two_gradcam_results.loc[
        ~two_gradcam_results["correct"]
    ]
    .sort_values(
        by="confidence",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Test:", len(two_gradcam_results))
print(
    "Corrette:",
    int(
        two_gradcam_results[
            "correct"
        ].sum()
    )
)
print("Errate:", len(two_errors))

display(two_errors)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix
)

class_names = list(test_loader_two.dataset.classes)

class_precision, class_recall, class_f1, class_support = (
    precision_recall_fscore_support(
        two_test_labels,
        two_test_predictions,
        labels=np.arange(len(class_names)),
        average=None,
        zero_division=0
    )
)

two_confusion_matrix = confusion_matrix(
    two_test_labels,
    two_test_predictions,
    labels=np.arange(len(class_names))
)

two_confusion_matrix_normalized = confusion_matrix(
    two_test_labels,
    two_test_predictions,
    labels=np.arange(len(class_names)),
    normalize="true"
)

two_classification_report = pd.DataFrame({
    "class": class_names,
    "precision": class_precision,
    "recall": class_recall,
    "f1_score": class_f1,
    "support": class_support.astype(int)
})

two_classification_report.to_csv(
    TWO_BRANCH_RESULTS / "test_classification_report.csv",
    index=False
)

pd.DataFrame(
    two_confusion_matrix,
    index=class_names,
    columns=class_names
).to_csv(
    TWO_BRANCH_RESULTS / "test_confusion_matrix.csv"
)

overall_names = [
    "Accuracy",
    "Balanced\nAccuracy",
    "Macro\nPrecision",
    "Macro\nRecall",
    "Macro F1",
    "Macro\nROC-AUC"
]

overall_values = [
    two_final_metrics["test_accuracy"],
    two_final_metrics["test_balanced_accuracy"],
    two_final_metrics["test_macro_precision"],
    two_final_metrics["test_macro_recall"],
    two_final_metrics["test_macro_f1"],
    two_final_metrics["test_macro_roc_auc_ovr"]
]

validation_f1 = two_final_metrics[
    "best_validation_macro_f1"
]

test_f1 = two_final_metrics[
    "test_macro_f1"
]

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(
    2,
    2,
    figsize=(18, 12)
)

fig.suptitle(
    "Valutazione finale della ResNet50 a due rami sul test set",
    fontsize=18,
    fontweight="bold"
)


colors = sns.color_palette(
    "muted",
    len(overall_values)
)

bars = axes[0, 0].bar(
    overall_names,
    overall_values,
    color=colors
)

axes[0, 0].set_title(
    "Metriche complessive sul test set"
)

axes[0, 0].set_ylabel("Valore")
axes[0, 0].set_ylim(0, 1.08)

axes[0, 0].bar_label(
    bars,
    fmt="%.3f",
    padding=3
)


x = np.arange(len(class_names))
bar_width = 0.25

bars_precision = axes[0, 1].bar(
    x - bar_width,
    class_precision,
    width=bar_width,
    label="Precision"
)

bars_recall = axes[0, 1].bar(
    x,
    class_recall,
    width=bar_width,
    label="Recall"
)

bars_f1 = axes[0, 1].bar(
    x + bar_width,
    class_f1,
    width=bar_width,
    label="F1-score"
)

axes[0, 1].set_title(
    "Metriche per classe sul test set"
)

axes[0, 1].set_ylabel("Valore")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(class_names)
axes[0, 1].set_ylim(0, 1.10)
axes[0, 1].legend()

for container in (
    bars_precision,
    bars_recall,
    bars_f1
):
    axes[0, 1].bar_label(
        container,
        fmt="%.2f",
        padding=2,
        fontsize=8,
        rotation=90
    )


comparison_names = [
    "Migliore validation\nMacro-F1",
    "Test\nMacro-F1"
]

comparison_values = [
    validation_f1,
    test_f1
]

comparison_bars = axes[1, 0].bar(
    comparison_names,
    comparison_values,
    color=["#76b7b2", "#e15759"]
)

axes[1, 0].set_title(
    "Confronto validation–test"
)

axes[1, 0].set_ylabel("Macro F1-score")
axes[1, 0].set_ylim(0, 1.08)

axes[1, 0].bar_label(
    comparison_bars,
    fmt="%.4f",
    padding=3
)

difference = test_f1 - validation_f1

axes[1, 0].text(
    0.5,
    0.08,
    f"Differenza test − validation: {difference:+.4f}",
    transform=axes[1, 0].transAxes,
    horizontalalignment="center",
    bbox={
        "boxstyle": "round",
        "facecolor": "white",
        "alpha": 0.9
    }
)


sns.heatmap(
    two_confusion_matrix_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[1, 1]
)

axes[1, 1].set_title(
    "Matrice di confusione normalizzata"
)

axes[1, 1].set_xlabel("Classe predetta")
axes[1, 1].set_ylabel("Classe reale")

plt.tight_layout(
    rect=[0, 0, 1, 0.96]
)

two_evaluation_png = (
    TWO_BRANCH_RESULTS
    / "two_branch_test_evaluation.png"
)

two_evaluation_pdf = (
    TWO_BRANCH_RESULTS
    / "two_branch_test_evaluation.pdf"
)

plt.savefig(
    two_evaluation_png,
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    two_evaluation_pdf,
    bbox_inches="tight"
)

plt.show()

print("\nGrafici salvati in:")
print(two_evaluation_png)
print(two_evaluation_pdf)


In [ ]:
plt.figure(figsize=(8, 7))

sns.heatmap(
    two_confusion_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(
    "Matrice di confusione della ResNet50 a due rami"
)
plt.xlabel("Classe predetta")
plt.ylabel("Classe reale")
plt.tight_layout()

confusion_path = (
    TWO_BRANCH_RESULTS
    / "two_branch_confusion_matrix.png"
)

plt.savefig(
    confusion_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Matrice salvata in:", confusion_path)


In [ ]:
import pandas as pd

rows = []

for split, dataset in {
    "Training": train_dataset_two,
    "Validation": val_dataset_two,
    "Test": test_dataset_two
}.items():

    for _, label in dataset.samples:
        rows.append({
            "split": split,
            "class": dataset.classes[label]
        })

distribution_table = pd.crosstab(
    pd.DataFrame(rows)["class"],
    pd.DataFrame(rows)["split"],
    margins=True,
    margins_name="Totale"
)

display(distribution_table)


## 8. Interpretabilità mediante Grad-CAM


In [ ]:
import torch.nn.functional as F


class TwoBranchGradCAM:

    def __init__(
        self,
        model,
        target_layer
    ):
        self.model = model
        self.activations = None
        self.gradients = None

        self.forward_handle = (
            target_layer.register_forward_hook(
                self._forward_hook
            )
        )

    def _forward_hook(
        self,
        module,
        inputs,
        output
    ):
        self.activations = output

        output.register_hook(
            self._save_gradients
        )

    def _save_gradients(
        self,
        gradients
    ):
        self.gradients = gradients

    def generate(
        self,
        left_tensor,
        right_tensor,
        valid_mask,
        target_class
    ):
        self.model.zero_grad(
            set_to_none=True
        )

        logits = self.model(
            left_tensor,
            right_tensor,
            valid_mask
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        score = logits[
            :,
            target_class
        ].sum()

        score.backward()

        activations = self.activations
        gradients = self.gradients

        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        cams = (
            weights * activations
        ).sum(
            dim=1,
            keepdim=True
        )

        cams = torch.relu(cams)

        cams = F.interpolate(
            cams,
            size=(
                KIDNEY_IMAGE_SIZE,
                KIDNEY_IMAGE_SIZE
            ),
            mode="bilinear",
            align_corners=False
        )

        left_cam = cams[0, 0]
        right_cam = cams[1, 0]

        combined_min = torch.minimum(
            left_cam.min(),
            right_cam.min()
        )

        combined_max = torch.maximum(
            left_cam.max(),
            right_cam.max()
        )

        denominator = (
            combined_max - combined_min
        ).clamp(min=1e-8)

        left_cam = (
            left_cam - combined_min
        ) / denominator

        right_cam = (
            right_cam - combined_min
        ) / denominator

        if not bool(valid_mask[0, 0]):
            left_cam = torch.zeros_like(
                left_cam
            )

        if not bool(valid_mask[0, 1]):
            right_cam = torch.zeros_like(
                right_cam
            )

        return {
            "left_heatmap": (
                left_cam.detach().cpu().numpy()
            ),
            "right_heatmap": (
                right_cam.detach().cpu().numpy()
            ),
            "probabilities": (
                probabilities
                .detach()
                .cpu()
                .numpy()[0]
            ),
            "logits": logits.detach().cpu()
        }

    def remove(self):
        self.forward_handle.remove()


two_grad_cam = TwoBranchGradCAM(
    model=model_two,
    target_layer=(
        model_two.encoder.layer4[-1].conv3
    )
)

print(
    "Grad-CAM inizializzata su:",
    model_two.encoder.layer4[-1].conv3
)


In [ ]:
def load_two_branch_case(row):

    matching_indices = [
        index
        for index, (path, _)
        in enumerate(
            test_dataset_two.samples
        )
        if path == row["path"]
    ]

    assert matching_indices, (
        f"Immagine non trovata: {row['path']}"
    )

    dataset_index = matching_indices[0]

    (
        left_tensor,
        right_tensor,
        valid_mask,
        label
    ) = test_dataset_two[dataset_index]

    return (
        left_tensor.unsqueeze(0).to(DEVICE),
        right_tensor.unsqueeze(0).to(DEVICE),
        valid_mask.unsqueeze(0).to(DEVICE),
        label
    )


In [ ]:
def tensor_to_display_image(tensor):

    mean = torch.tensor(
        IMAGENET_MEAN,
        device=tensor.device
    ).view(3, 1, 1)

    std = torch.tensor(
        IMAGENET_STD,
        device=tensor.device
    ).view(3, 1, 1)

    image = (
        tensor * std + mean
    ).clamp(0, 1)

    return (
        image
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .numpy()
    )


def create_overlay_two_branch(
    image,
    heatmap,
    alpha=0.45
):
    colored_heatmap = plt.get_cmap(
        "jet"
    )(heatmap)[..., :3]

    overlay = (
        (1 - alpha) * image
        + alpha * colored_heatmap
    )

    return np.clip(
        overlay,
        0,
        1
    )


def show_two_branch_gradcam(
    row,
    save_directory,
    target="predicted"
):
    (
        left_tensor,
        right_tensor,
        valid_mask,
        _
    ) = load_two_branch_case(row)

    if target == "true":
        target_index = int(
            row["true_index"]
        )
    else:
        target_index = int(
            row["predicted_index"]
        )

    output = two_grad_cam.generate(
        left_tensor=left_tensor,
        right_tensor=right_tensor,
        valid_mask=valid_mask,
        target_class=target_index
    )

    left_image = tensor_to_display_image(
        left_tensor[0]
    )

    right_image = tensor_to_display_image(
        right_tensor[0]
    )

    left_heatmap = output[
        "left_heatmap"
    ]

    right_heatmap = output[
        "right_heatmap"
    ]

    left_overlay = create_overlay_two_branch(
        left_image,
        left_heatmap
    )

    right_overlay = create_overlay_two_branch(
        right_image,
        right_heatmap
    )

    probabilities = output[
        "probabilities"
    ]

    predicted_index = int(
        row["predicted_index"]
    )

    left_score = float(
        left_heatmap.mean()
    )

    right_score = float(
        right_heatmap.mean()
    )

    total_score = (
        left_score + right_score
    )

    if total_score > 0:
        left_percentage = (
            left_score / total_score * 100
        )

        right_percentage = (
            right_score / total_score * 100
        )
    else:
        left_percentage = 0.0
        right_percentage = 0.0

    figure, axes = plt.subplots(
        2,
        2,
        figsize=(10, 10)
    )

    axes[0, 0].imshow(left_image)
    axes[0, 0].set_title("Rene 1")

    axes[0, 1].imshow(right_image)
    axes[0, 1].set_title("Rene 2")

    axes[1, 0].imshow(left_overlay)
    axes[1, 0].set_title(
        f"Grad-CAM rene 1\n"
        f"Attenzione: {left_percentage:.1f}%"
    )

    axes[1, 1].imshow(right_overlay)
    axes[1, 1].set_title(
        f"Grad-CAM rene 2\n"
        f"Attenzione: {right_percentage:.1f}%"
    )

    for axis in axes.flatten():
        axis.axis("off")

    figure.suptitle(
        f"{Path(row['path']).name}\n"
        f"Reale: {row['true_class']} | "
        f"Predetta: {row['predicted_class']} | "
        f"Confidenza: "
        f"{probabilities[predicted_index]:.3f}\n"
        f"Grad-CAM per classe: "
        f"{CLASS_NAMES[target_index]}",
        fontsize=12
    )

    plt.tight_layout()

    save_directory = Path(
        save_directory
    )

    save_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    output_path = (
        save_directory
        / (
            f"{row['true_class']}_as_"
            f"{row['predicted_class']}_"
            f"{Path(row['path']).stem}_"
            f"target_{CLASS_NAMES[target_index]}.png"
        )
    )

    plt.savefig(
        output_path,
        dpi=250,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(figure)

    return output_path


In [ ]:
TWO_GRADCAM_ERRORS = (
    TWO_BRANCH_RESULTS
    / "gradcam_errors"
)

for _, row in two_errors.head(12).iterrows():

    show_two_branch_gradcam(
        row=row,
        save_directory=TWO_GRADCAM_ERRORS,
        target="predicted"
    )


In [ ]:
two_correct = two_gradcam_results.loc[
    two_gradcam_results["correct"]
].copy()

two_correct_cases = []

for class_name in CLASS_NAMES:

    selected = (
        two_correct.loc[
            two_correct["true_class"]
            == class_name
        ]
        .sort_values(
            by="confidence",
            ascending=False
        )
        .head(3)
    )

    two_correct_cases.append(
        selected
    )

two_correct_cases = pd.concat(
    two_correct_cases,
    ignore_index=True
)

TWO_GRADCAM_CORRECT = (
    TWO_BRANCH_RESULTS
    / "gradcam_correct"
)

for _, row in two_correct_cases.iterrows():

    show_two_branch_gradcam(
        row=row,
        save_directory=TWO_GRADCAM_CORRECT,
        target="predicted"
    )


In [ ]:
two_grad_cam.remove()
print("Hook Grad-CAM rimosso correttamente")


## 9. Fine dell'esperimento

I checkpoint sono selezionati esclusivamente mediante il Macro-F1 sul validation set. Il test set viene utilizzato soltanto per la valutazione finale e l'analisi post-hoc.
